[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/04_diseno_tokenizers_modernos/04_diseno_tokenizers_modernos.ipynb)

# Módulo 04 · Diseño de tokenizers modernos: vocabulario, multilingüe y costes

| | |
|---|---|
| **Nivel** | 🟠 Avanzado |
| **Duración estimada** | ≈ 5 h (lección ≈ 3 h + proyecto ≈ 2 h) |
| **GPU recomendada** | Ninguna (CPU). Coste en Colab ≈ 0 unidades |
| **Prerrequisitos** | Módulos 01–03 (bytes, BPE, WordPiece/Unigram/SentencePiece, `tiktoken`, HF `tokenizers`) |
| **Hilo conductor** | **Políglota**: decides el vocabulario del buscador (ES/EN/código) con criterios de ingeniería, no por costumbre |

> **Contexto Políglota Labs.** Ya sabes *entrenar* tokenizers (módulo 03). Ahora te preguntan: «¿de qué tamaño?, ¿qué mezcla de idiomas?, ¿cuánto nos cuesta que un cliente escriba en español en vez de inglés?,
> ¿cómo tokenizamos números e indentación?, ¿qué tokens especiales necesitamos para el chat del asistente?». Esta lección convierte esas preguntas en **medidas y fórmulas**.

## 1. Objetivos de aprendizaje

1. **Formular** el diseño de un vocabulario como un problema de presupuesto (parámetros de embedding, coste del softmax, longitud de secuencia) y **calcular** sus trade-offs.
2. **Resumir** qué dicen las leyes de escala con el vocabulario (Tao et al., 2024) y por qué los vocabularios de los modelos recientes crecieron.
3. **Comparar** vocabularios reales (GPT-2/GPT-4/GPT-4o, Llama, Gemma, Qwen, Mistral/Tekken) con cifras **verificadas y fechadas** y reconocer qué cifras no están verificadas.
4. **Medir** fertilidad, bytes por token y **paridad entre idiomas** («token tax», prima de tokenización) sobre texto paralelo, y **traducirla a dinero y a contexto útil**.
5. **Analizar** cómo se tokenizan números, indentación y código, y qué diseño (dígitos sueltos, grupos, R2L, espacios) cambia el rendimiento.
6. **Diseñar** tokens especiales y *chat templates* correctamente (y evitar los errores clásicos: doble BOS, máscaras de pérdida, plantillas distintas entre entrenamiento y servicio).
7. **Implementar** una versión mínima de **SuperBPE** (tokens que cruzan palabras) y evaluar su ganancia y sus límites.
8. **Decidir** el vocabulario de Políglota con una función de coste explícita (calidad por idioma vs memoria).

## 2. 🔁 Conexión con módulos anteriores

<details><summary><b>R1 · (Módulo 03)</b> ¿Cuál es la diferencia entre fertilidad y bytes por token, y por qué no basta la fertilidad para comparar idiomas como el chino o el japonés?</summary>

Fertilidad = tokens/palabra; bytes por token = bytes UTF-8 de texto cubiertos por cada token. En idiomas sin espacios entre palabras (zh, ja) «palabra» no está bien definida, así que la fertilidad no es comparable; hay que normalizar por **contenido paralelo** (la misma frase traducida) o por bytes/caracteres.
</details>

<details><summary><b>R2 · (Módulo 02)</b> ¿Por qué un tokenizer byte-level no tiene <code>[UNK]</code> pero puede ser carísimo para ciertos idiomas?</summary>

Cualquier texto se descompone en bytes, así que siempre es representable. Pero un carácter no latino ocupa 2–4 bytes en UTF-8 y, si el vocabulario no tiene merges para esa escritura, cada byte es un token: tokens por carácter ≈ bytes por carácter.
</details>

<details><summary><b>R3 · (Módulo 03)</b> ¿Qué decide el pre-tokenizador y por qué ningún token puede cruzar sus fronteras?</summary>

Define las unidades dentro de las que se aplican los merges; si dos símbolos están en pre-tokens distintos nunca se fusionan. Hoy veremos qué pasa si se **relaja** esa restricción (SuperBPE).
</details>

In [ ]:
!pip install -q -U tokenizers sentencepiece tiktoken datasets matplotlib seaborn scikit-learn

In [ ]:
import os, sys, re, math, json, random, time, unicodedata, collections, itertools, warnings
os.environ.setdefault("OMP_NUM_THREADS", "1")          # evita sobre-suscripción de hilos
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
FAST_DEV_RUN = True      # True: tamaños diminutos (CPU, minutos). False: escala completa (Colab Pro)
random.seed(SEED); np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False})
COL = {"azul": "#1F6FEB", "naranja": "#F28E2B", "verde": "#2CA02C", "gris": "#6B7280", "rojo": "#D62728"}
try:
    import torch
    torch.set_num_threads(1); torch.manual_seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    device = "cpu"
print("device:", device, "| FAST_DEV_RUN:", FAST_DEV_RUN)

### Utilidades de datos

Copia local de `carga_corpus()` (contrato común; ver módulo 03/00) y de `carga_paralelo()`: **frases paralelas** para medir paridad. Cadena para el paralelo: **FLORES+** (`openlanguagedata/flores_plus`, puede requerir aceptar términos y un token de HF) →
**opus-100 en-es** → **mini-conjunto de 16 frases × 12 idiomas escrito a mano** (aviso en pantalla). Con el mini-conjunto las cifras absolutas son *orientativas* (n pequeño): las conclusiones relativas (qué idioma paga más) son las que se sostienen.

In [ ]:
# ---- Utilidades de corpus Políglota (copia local; contrato: carga_corpus(fast_dev) -> dict) ----
import urllib.request, urllib.error, zipfile, io, inspect, ast

def hay_red(url: str, timeout: float = 6.0) -> bool:
    """Sonda rápida para no esperar minutos si el host está bloqueado (proxy corporativo, etc.)."""
    try:
        urllib.request.urlopen(urllib.request.Request(url, method="HEAD"), timeout=timeout)
        return True
    except urllib.error.HTTPError:
        return True          # el servidor respondió (aunque sea 4xx): hay ruta de red
    except Exception:
        return False         # túnel bloqueado, DNS, timeout...

def _hf_wikipedia(lang: str, n: int, max_chars: int):
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
    out = []
    for ex in ds:
        if len(ex["text"]) > 400:
            out.append({"id": f"wiki_{lang}_{len(out)}", "lang": lang, "text": ex["text"][:max_chars]})
        if len(out) >= n:
            break
    return out

def _hf_codesearchnet(n: int):
    from datasets import load_dataset
    for repo, kw in [("code-search-net/code_search_net", {"name": "python"}), ("claudios/code_search_net", {"name": "python"}),
                     ("Nan-Do/code-search-net-python", {})]:
        try:
            ds = load_dataset(repo, split="train", streaming=True, **kw)
            out = []
            for ex in ds:
                code = ex.get("func_code_string") or ex.get("code") or ex.get("original_string")
                if code and 80 < len(code) < 3000:
                    out.append({"id": f"csn_{len(out)}", "lang": "code", "text": code})
                if len(out) >= n:
                    return out
        except Exception as e:
            print("   (CodeSearchNet vía", repo, "falló:", type(e).__name__, ")")
    raise RuntimeError("CodeSearchNet no disponible")

def _detok(s: str) -> str:
    s = re.sub(r"\s+([,.;:!?%)\]])", r"\1", s)
    s = re.sub(r"([(\[¿¡])\s+", r"\1", s)
    return re.sub(r"\s+", " ", s).strip()

In [ ]:
def _espejo_nltk(n: int):
    """Espejo público: muestra de Europarl (ES/EN) del repositorio nltk_data en GitHub (CC/uso de investigación)."""
    url = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/corpora/europarl_raw.zip"
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url, timeout=60).read()))
    docs = []
    for lang, folder, ext in [("es", "spanish", ".es"), ("en", "english", ".en")]:
        lines = []
        for name in sorted(z.namelist()):
            if f"/{folder}/" in name and name.endswith(ext):
                lines += [_detok(l) for l in z.read(name).decode("utf-8", "ignore").splitlines() if len(l.split()) > 6]
        step = 8
        chunks = [" ".join(lines[i:i + step]) for i in range(0, len(lines), step)]
        random.Random(SEED).shuffle(chunks)
        docs += [{"id": f"ep_{lang}_{i}", "lang": lang, "text": c[:2000]} for i, c in enumerate(chunks[:n])]
    return docs

def _codigo_stdlib(n: int):
    """Código Python real de la biblioteca estándar (siempre disponible, sin red)."""
    import json, re as _re, argparse, textwrap, collections as _c, itertools as _i, functools, pathlib, csv, heapq, bisect
    import statistics, difflib, string, shutil, tempfile, fractions, random as _r, datetime, calendar, glob, fnmatch, logging
    out = []
    for mod in [json.decoder, json.encoder, _re, argparse, textwrap, functools, pathlib, csv, heapq, bisect, statistics, difflib,
                shutil, tempfile, fractions, datetime, calendar, glob, fnmatch, logging, _c, _r]:
        try:
            tree = ast.parse(inspect.getsource(mod)); src = inspect.getsource(mod).splitlines()
        except (OSError, TypeError, SyntaxError):
            continue
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef,)) and 4 <= (node.end_lineno - node.lineno) <= 40:
                code = "\n".join(src[node.lineno - 1: node.end_lineno])
                if 80 < len(code) < 3000:
                    out.append({"id": f"std_{len(out)}", "lang": "code", "text": code})
    random.Random(SEED).shuffle(out)
    return out[:n]

In [ ]:
_ES_S = ["el sistema", "la base de datos", "el servidor", "la aplicación", "el modelo", "la biblioteca", "el usuario", "la función", "el módulo", "la consulta"]
_ES_V = ["procesa", "almacena", "devuelve", "actualiza", "valida", "descarga", "ejecuta", "registra", "indexa", "compara"]
_ES_O = ["los documentos relevantes", "una lista de resultados", "los parámetros de configuración", "las peticiones entrantes", "la información almacenada",
         "varias tablas de forma eficiente", "los errores más frecuentes", "las versiones anteriores", "los textos en diferentes idiomas", "un conjunto de datos nuevo"]
_ES_A = ["rápidamente", "automáticamente", "correctamente", "de manera incremental", "sin interrupciones", "cada vez que cambia la configuración", "antes de la instalación", "durante la inicialización"]
_EN_S = ["the system", "the database", "the server", "the application", "the model", "the library", "the user", "the function", "the module", "the query"]
_EN_V = ["processes", "stores", "returns", "updates", "validates", "downloads", "executes", "logs", "indexes", "compares"]
_EN_O = ["the relevant documents", "a list of results", "the configuration parameters", "the incoming requests", "the stored information",
         "several tables efficiently", "the most frequent errors", "the previous versions", "texts in different languages", "a new dataset"]
_EN_A = ["quickly", "automatically", "correctly", "incrementally", "without interruptions", "whenever the configuration changes", "before the installation", "during initialization"]

def _sintetico(n: int):
    """Último recurso: texto ES/EN generado por plantillas + funciones Python plantilla (mismo esquema)."""
    rng = random.Random(SEED); docs = []
    for lang, S, V, O, A in [("es", _ES_S, _ES_V, _ES_O, _ES_A), ("en", _EN_S, _EN_V, _EN_O, _EN_A)]:
        for i in range(n):
            sents = []
            for _ in range(rng.randint(4, 8)):
                s = f"{rng.choice(S)} {rng.choice(V)} {rng.choice(O)} {rng.choice(A)}."
                sents.append(s[0].upper() + s[1:])
            docs.append({"id": f"syn_{lang}_{i}", "lang": lang, "text": " ".join(sents)})
    names = ["load", "parse", "merge", "filter", "count", "split", "encode", "decode", "score", "rank"]
    things = ["items", "tokens", "rows", "words", "records", "paths", "values", "pairs"]
    for i in range(n):
        f, t = rng.choice(names), rng.choice(things)
        docs.append({"id": f"syn_code_{i}", "lang": "code", "text":
            f"def {f}_{t}({t}, limit={rng.randint(2, 99)}):\n    \"\"\"{f.capitalize()} the {t} and return at most limit results.\"\"\"\n"
            f"    result = []\n    for x in {t}:\n        if len(result) >= limit:\n            break\n        result.append(x)\n    return result\n"})
    return docs

In [ ]:
def _mini_golden(docs, n_q: int = 24):
    """Mini golden set: la consulta es la primera frase (≤ 12 palabras) del documento; qrels {qid: {doc_id: 1}}."""
    rng = random.Random(SEED); queries, qrels = [], {}
    for lang in ["es", "en", "code"]:
        pool = [d for d in docs if d["lang"] == lang]; rng.shuffle(pool)
        for d in pool[: n_q // 3]:
            if lang == "code":
                try:
                    q = (ast.get_docstring(ast.parse(d["text"]).body[0]) or "").split("\n")[0]
                except SyntaxError:
                    q = ""
                q = q or re.sub(r"[_(].*", "", d["text"].split("def ")[-1]).replace("_", " ")
            else:
                q = " ".join(re.split(r"(?<=[.!?])\s", d["text"])[0].split()[:12])
            qid = f"q{len(queries)}"; queries.append({"qid": qid, "lang": lang, "text": q}); qrels[qid] = {d["id"]: 1}
    return queries, qrels

In [ ]:
def carga_corpus(fast_dev: bool = True) -> dict:
    """Corpus Políglota. Cadena: oficial (HF Wikipedia + CodeSearchNet) -> espejo (Europarl vía GitHub + stdlib) -> sintético.
    Devuelve {"docs", "queries", "qrels", "source": "real"|"synthetic", "detail": {...}}."""
    n = 150 if fast_dev else 3000
    max_chars = 2000 if fast_dev else 6000
    docs, detail = [], {}
    hf_ok = hay_red("https://huggingface.co")
    for lang in ["es", "en"]:
        got = None
        if hf_ok:
            try: got, detail[lang] = _hf_wikipedia(lang, n, max_chars), "hf:wikimedia/wikipedia"
            except Exception as e: print(f"   (Wikipedia {lang} falló: {type(e).__name__})")
        if got is None and hay_red("https://raw.githubusercontent.com"):
            try:
                mirror = _espejo_nltk(n); got = [d for d in mirror if d["lang"] == lang]; detail[lang] = "espejo:nltk-europarl"
            except Exception as e: print(f"   (espejo {lang} falló: {type(e).__name__})")
        if got is None: got, detail[lang] = [d for d in _sintetico(n) if d["lang"] == lang], "sintetico"
        docs += got
    code = None
    if hf_ok:
        try: code, detail["code"] = _hf_codesearchnet(n), "hf:code_search_net"
        except Exception as e: print(f"   (CodeSearchNet falló: {type(e).__name__})")
    if code is None:
        code = _codigo_stdlib(n); detail["code"] = "espejo:stdlib-python"
        if len(code) < 10: code, detail["code"] = [d for d in _sintetico(n) if d["lang"] == "code"], "sintetico"
    docs += code
    queries, qrels = _mini_golden(docs)
    real = all(not v.startswith("sintetico") for v in detail.values())
    if not real: print("AVISO: parte del corpus es SINTÉTICA (sin acceso a los datasets). Las cifras absolutas cambiarán con datos reales.")
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": "real" if real else "synthetic", "detail": detail}

In [ ]:
corpus = carga_corpus(fast_dev=FAST_DEV_RUN)
docs = corpus["docs"]
print("fuente:", corpus["source"], corpus["detail"])
by_lang = {l: [d["text"] for d in docs if d["lang"] == l] for l in ["es", "en", "code"]}
for l, ts in by_lang.items():
    print(f"{l:>4}: {len(ts):4d} docs | {sum(len(t) for t in ts)/1e3:8.1f} k caracteres | {sum(len(t.encode()) for t in ts)/1e3:8.1f} k bytes")
print("\nEjemplo ES:", by_lang["es"][0][:160].replace("\n", " "))
print("Ejemplo código:\n", by_lang["code"][0][:200])

In [ ]:
# Frases paralelas (tipo FLORES): oficial -> espejo -> mini-conjunto escrito a mano (16 frases x 12 idiomas)
_PARALELO_MANO = {
 "en": ["The search engine returns the most relevant documents for each query.", "Please restart the server before installing the new version.",
        "The model was trained on a large collection of texts in several languages.", "Every user must change their password every ninety days.",
        "The function receives a list of numbers and returns their average.", "Yesterday it rained all day, so we stayed at home.",
        "The company opened three new offices in Europe this year.", "Reading the documentation carefully saves a lot of time.",
        "The meeting will begin at nine o'clock in the morning.", "Our system supports more than forty languages and several file formats.",
        "The children are playing in the park near the river.", "To install the library, run the command shown below.",
        "The results show that the new method is faster and more accurate.", "Water boils at one hundred degrees Celsius at sea level.",
        "She asked whether the database could be restored from a backup.", "Errors must be recorded in a log file for later analysis."],
 "es": ["El motor de búsqueda devuelve los documentos más relevantes para cada consulta.", "Reinicie el servidor antes de instalar la nueva versión.",
        "El modelo se entrenó con una gran colección de textos en varios idiomas.", "Cada usuario debe cambiar su contraseña cada noventa días.",
        "La función recibe una lista de números y devuelve su promedio.", "Ayer llovió todo el día, así que nos quedamos en casa.",
        "La empresa abrió tres oficinas nuevas en Europa este año.", "Leer la documentación con atención ahorra mucho tiempo.",
        "La reunión comenzará a las nueve de la mañana.", "Nuestro sistema admite más de cuarenta idiomas y varios formatos de archivo.",
        "Los niños juegan en el parque cerca del río.", "Para instalar la biblioteca, ejecute el comando que se muestra a continuación.",
        "Los resultados muestran que el nuevo método es más rápido y más preciso.", "El agua hierve a cien grados Celsius al nivel del mar.",
        "Ella preguntó si la base de datos se podía restaurar desde una copia de seguridad.", "Los errores deben registrarse en un archivo de registro para su análisis posterior."],
 "fr": ["Le moteur de recherche renvoie les documents les plus pertinents pour chaque requête.", "Veuillez redémarrer le serveur avant d'installer la nouvelle version.",
        "Le modèle a été entraîné sur une grande collection de textes dans plusieurs langues.", "Chaque utilisateur doit changer son mot de passe tous les quatre-vingt-dix jours.",
        "La fonction reçoit une liste de nombres et renvoie leur moyenne.", "Hier, il a plu toute la journée, alors nous sommes restés à la maison.",
        "L'entreprise a ouvert trois nouveaux bureaux en Europe cette année.", "Lire attentivement la documentation fait gagner beaucoup de temps.",
        "La réunion commencera à neuf heures du matin.", "Notre système prend en charge plus de quarante langues et plusieurs formats de fichier.",
        "Les enfants jouent dans le parc près de la rivière.", "Pour installer la bibliothèque, exécutez la commande ci-dessous.",
        "Les résultats montrent que la nouvelle méthode est plus rapide et plus précise.", "L'eau bout à cent degrés Celsius au niveau de la mer.",
        "Elle a demandé si la base de données pouvait être restaurée à partir d'une sauvegarde.", "Les erreurs doivent être enregistrées dans un fichier journal pour une analyse ultérieure."],
 "de": ["Die Suchmaschine liefert für jede Anfrage die relevantesten Dokumente.", "Bitte starten Sie den Server neu, bevor Sie die neue Version installieren.",
        "Das Modell wurde mit einer großen Sammlung von Texten in mehreren Sprachen trainiert.", "Jeder Benutzer muss sein Passwort alle neunzig Tage ändern.",
        "Die Funktion erhält eine Liste von Zahlen und gibt deren Durchschnitt zurück.", "Gestern hat es den ganzen Tag geregnet, deshalb sind wir zu Hause geblieben.",
        "Das Unternehmen hat dieses Jahr drei neue Büros in Europa eröffnet.", "Wer die Dokumentation sorgfältig liest, spart viel Zeit.",
        "Die Besprechung beginnt um neun Uhr morgens.", "Unser System unterstützt mehr als vierzig Sprachen und mehrere Dateiformate.",
        "Die Kinder spielen im Park in der Nähe des Flusses.", "Um die Bibliothek zu installieren, führen Sie den unten gezeigten Befehl aus.",
        "Die Ergebnisse zeigen, dass die neue Methode schneller und genauer ist.", "Wasser kocht auf Meereshöhe bei hundert Grad Celsius.",
        "Sie fragte, ob die Datenbank aus einer Sicherung wiederhergestellt werden könne.", "Fehler müssen für die spätere Analyse in einer Protokolldatei festgehalten werden."],
 "pt": ["O mecanismo de busca devolve os documentos mais relevantes para cada consulta.", "Reinicie o servidor antes de instalar a nova versão.",
        "O modelo foi treinado com uma grande coleção de textos em vários idiomas.", "Cada usuário deve trocar a senha a cada noventa dias.",
        "A função recebe uma lista de números e devolve a média deles.", "Ontem choveu o dia todo, por isso ficamos em casa.",
        "A empresa abriu três novos escritórios na Europa este ano.", "Ler a documentação com atenção poupa muito tempo.",
        "A reunião começará às nove horas da manhã.", "Nosso sistema suporta mais de quarenta idiomas e vários formatos de arquivo.",
        "As crianças estão brincando no parque perto do rio.", "Para instalar a biblioteca, execute o comando mostrado abaixo.",
        "Os resultados mostram que o novo método é mais rápido e mais preciso.", "A água ferve a cem graus Celsius ao nível do mar.",
        "Ela perguntou se o banco de dados poderia ser restaurado a partir de um backup.", "Os erros devem ser registrados em um arquivo de log para análise posterior."],
}

In [ ]:
_PARALELO_MANO.update({
 "ru": ["Поисковая система возвращает наиболее релевантные документы для каждого запроса.", "Перезапустите сервер перед установкой новой версии.",
        "Модель была обучена на большой коллекции текстов на нескольких языках.", "Каждый пользователь должен менять пароль каждые девяносто дней.",
        "Функция принимает список чисел и возвращает их среднее значение.", "Вчера целый день шёл дождь, поэтому мы остались дома.",
        "В этом году компания открыла три новых офиса в Европе.", "Внимательное чтение документации экономит много времени.",
        "Совещание начнётся в девять часов утра.", "Наша система поддерживает более сорока языков и несколько форматов файлов.",
        "Дети играют в парке возле реки.", "Чтобы установить библиотеку, выполните команду, приведённую ниже.",
        "Результаты показывают, что новый метод быстрее и точнее.", "Вода кипит при ста градусах Цельсия на уровне моря.",
        "Она спросила, можно ли восстановить базу данных из резервной копии.", "Ошибки необходимо записывать в файл журнала для последующего анализа."],
 "zh": ["搜索引擎会为每个查询返回最相关的文档。", "请在安装新版本之前重新启动服务器。", "该模型是在多种语言的大量文本上训练的。", "每位用户必须每九十天更换一次密码。",
        "该函数接收一个数字列表并返回它们的平均值。", "昨天下了一整天的雨，所以我们待在家里。", "今年公司在欧洲新开了三个办事处。", "仔细阅读文档可以节省很多时间。",
        "会议将于上午九点开始。", "我们的系统支持四十多种语言和多种文件格式。", "孩子们正在河边的公园里玩耍。", "要安装该库，请运行下面显示的命令。",
        "结果表明，新方法更快、更准确。", "在海平面，水在一百摄氏度时沸腾。", "她问是否可以从备份中恢复数据库。", "错误必须记录在日志文件中，以便日后分析。"],
 "ja": ["検索エンジンは、各クエリに対して最も関連性の高い文書を返します。", "新しいバージョンをインストールする前に、サーバーを再起動してください。", "このモデルは、複数の言語による大量のテキストで訓練されました。",
        "すべてのユーザーは九十日ごとにパスワードを変更しなければなりません。", "この関数は数値のリストを受け取り、その平均を返します。", "昨日は一日中雨が降っていたので、私たちは家にいました。",
        "その会社は今年、ヨーロッパに三つの新しいオフィスを開きました。", "ドキュメントを丁寧に読むと、多くの時間を節約できます。", "会議は午前九時に始まります。",
        "私たちのシステムは、四十以上の言語と複数のファイル形式に対応しています。", "子どもたちは川の近くの公園で遊んでいます。", "ライブラリをインストールするには、以下のコマンドを実行してください。",
        "結果は、新しい方法のほうが速く、より正確であることを示しています。", "水は海面の高さで摂氏百度で沸騰します。", "彼女は、データベースをバックアップから復元できるかどうか尋ねました。",
        "エラーは、後で分析できるようにログファイルに記録する必要があります。"],
 "ko": ["검색 엔진은 각 질의에 대해 가장 관련성이 높은 문서를 반환합니다.", "새 버전을 설치하기 전에 서버를 다시 시작하십시오.", "이 모델은 여러 언어로 된 방대한 텍스트 모음으로 학습되었습니다.",
        "모든 사용자는 구십 일마다 비밀번호를 변경해야 합니다.", "이 함수는 숫자 목록을 받아 그 평균을 반환합니다.", "어제는 하루 종일 비가 와서 우리는 집에 있었습니다.",
        "그 회사는 올해 유럽에 새 사무실 세 곳을 열었습니다.", "문서를 꼼꼼히 읽으면 많은 시간을 절약할 수 있습니다.", "회의는 오전 아홉 시에 시작됩니다.",
        "저희 시스템은 마흔 개가 넘는 언어와 여러 파일 형식을 지원합니다.", "아이들이 강 근처 공원에서 놀고 있습니다.", "라이브러리를 설치하려면 아래에 표시된 명령을 실행하십시오.",
        "결과는 새로운 방법이 더 빠르고 더 정확하다는 것을 보여 줍니다.", "물은 해수면에서 섭씨 백 도에 끓습니다.", "그녀는 데이터베이스를 백업에서 복원할 수 있는지 물었습니다.",
        "오류는 나중에 분석할 수 있도록 로그 파일에 기록해야 합니다."],
 "ar": ["يعيد محرك البحث أكثر المستندات صلة بكل استعلام.", "يرجى إعادة تشغيل الخادم قبل تثبيت الإصدار الجديد.", "تم تدريب النموذج على مجموعة كبيرة من النصوص بعدة لغات.",
        "يجب على كل مستخدم تغيير كلمة المرور كل تسعين يومًا.", "تستقبل الدالة قائمة من الأرقام وتعيد متوسطها.", "أمطرت السماء طوال أمس، لذلك بقينا في المنزل.",
        "افتتحت الشركة ثلاثة مكاتب جديدة في أوروبا هذا العام.", "قراءة الوثائق بعناية توفر الكثير من الوقت.", "سيبدأ الاجتماع في الساعة التاسعة صباحًا.",
        "يدعم نظامنا أكثر من أربعين لغة وعدة تنسيقات للملفات.", "الأطفال يلعبون في الحديقة القريبة من النهر.", "لتثبيت المكتبة، شغّل الأمر الموضح أدناه.",
        "تُظهر النتائج أن الطريقة الجديدة أسرع وأكثر دقة.", "يغلي الماء عند مئة درجة مئوية عند مستوى سطح البحر.", "سألت عما إذا كان بالإمكان استعادة قاعدة البيانات من نسخة احتياطية.",
        "يجب تسجيل الأخطاء في ملف سجل لتحليلها لاحقًا."],
 "hi": ["खोज इंजन हर क्वेरी के लिए सबसे प्रासंगिक दस्तावेज़ लौटाता है।", "नया संस्करण इंस्टॉल करने से पहले कृपया सर्वर को पुनः आरंभ करें।", "इस मॉडल को कई भाषाओं के पाठों के एक बड़े संग्रह पर प्रशिक्षित किया गया था।",
        "हर उपयोगकर्ता को हर नब्बे दिन में अपना पासवर्ड बदलना होगा।", "यह फ़ंक्शन संख्याओं की एक सूची लेता है और उनका औसत लौटाता है।", "कल पूरे दिन बारिश हुई, इसलिए हम घर पर ही रहे।",
        "कंपनी ने इस साल यूरोप में तीन नए कार्यालय खोले।", "दस्तावेज़ीकरण को ध्यान से पढ़ने से बहुत समय बचता है।", "बैठक सुबह नौ बजे शुरू होगी।",
        "हमारा सिस्टम चालीस से अधिक भाषाओं और कई फ़ाइल प्रारूपों का समर्थन करता है।", "बच्चे नदी के पास वाले पार्क में खेल रहे हैं।", "लाइब्रेरी इंस्टॉल करने के लिए नीचे दिखाया गया कमांड चलाएँ।",
        "परिणाम दिखाते हैं कि नई विधि अधिक तेज़ और अधिक सटीक है।", "समुद्र तल पर पानी सौ डिग्री सेल्सियस पर उबलता है।", "उसने पूछा कि क्या डेटाबेस को बैकअप से बहाल किया जा सकता है।",
        "बाद में विश्लेषण के लिए त्रुटियों को लॉग फ़ाइल में दर्ज किया जाना चाहिए।"],
 "tr": ["Arama motoru her sorgu için en alakalı belgeleri döndürür.", "Yeni sürümü yüklemeden önce lütfen sunucuyu yeniden başlatın.", "Model, birçok dilde yazılmış büyük bir metin koleksiyonuyla eğitildi.",
        "Her kullanıcı parolasını doksan günde bir değiştirmelidir.", "İşlev bir sayı listesi alır ve bunların ortalamasını döndürür.", "Dün bütün gün yağmur yağdı, bu yüzden evde kaldık.",
        "Şirket bu yıl Avrupa'da üç yeni ofis açtı.", "Belgeleri dikkatle okumak çok zaman kazandırır.", "Toplantı sabah saat dokuzda başlayacak.",
        "Sistemimiz kırktan fazla dili ve birçok dosya biçimini destekliyor.", "Çocuklar nehrin yakınındaki parkta oynuyor.", "Kitaplığı yüklemek için aşağıda gösterilen komutu çalıştırın.",
        "Sonuçlar, yeni yöntemin daha hızlı ve daha doğru olduğunu gösteriyor.", "Su, deniz seviyesinde yüz santigrat derecede kaynar.", "Veritabanının bir yedekten geri yüklenip yüklenemeyeceğini sordu.",
        "Hatalar, daha sonra analiz edilebilmesi için bir günlük dosyasına kaydedilmelidir."],
})

In [ ]:
FLORES_CODES = {"en": "eng_Latn", "es": "spa_Latn", "fr": "fra_Latn", "de": "deu_Latn", "pt": "por_Latn", "ru": "rus_Cyrl", "zh": "zho_Hans",
                "ja": "jpn_Jpan", "ko": "kor_Hang", "ar": "arb_Arab", "hi": "hin_Deva", "tr": "tur_Latn"}

def carga_paralelo(fast_dev: bool = True, langs=None) -> dict:
    """Frases paralelas. Cadena: FLORES+ (HF, puede pedir aceptar términos y token) -> opus-100 en-es -> mini-conjunto a mano.
    Devuelve {"langs": {lang: [frases alineadas]}, "source": "flores+"|"opus-100"|"mano", "n": int}."""
    langs = langs or list(FLORES_CODES)
    n = 100 if fast_dev else 997
    if hay_red("https://huggingface.co"):
        try:
            from datasets import load_dataset
            out = {l: [ex["text"] for ex in load_dataset("openlanguagedata/flores_plus", FLORES_CODES[l], split="devtest")][:n] for l in langs}
            return {"langs": out, "source": "flores+", "n": len(out["en"])}
        except Exception as e:
            print("   (FLORES+ no disponible:", type(e).__name__, "-> intento opus-100 en-es)")
        try:
            from datasets import load_dataset
            ds = load_dataset("Helsinki-NLP/opus-100", "en-es", split="test")
            tr = [ex["translation"] for ex in ds][:n]
            return {"langs": {"en": [t["en"] for t in tr], "es": [t["es"] for t in tr]}, "source": "opus-100", "n": len(tr)}
        except Exception as e:
            print("   (opus-100 no disponible:", type(e).__name__, ")")
    print("AVISO: usando el mini-conjunto paralelo escrito a mano (16 frases). Para cifras fiables usa FLORES+ con acceso a HF.")
    out = {l: _PARALELO_MANO[l] for l in langs if l in _PARALELO_MANO}
    return {"langs": out, "source": "mano", "n": len(out["en"])}

par = carga_paralelo(fast_dev=FAST_DEV_RUN)
print('paralelo:', par['source'], '| frases:', par['n'], '| idiomas:', list(par['langs']))

In [ ]:
# ── Métricas de retrieval canónicas de Políglota (módulo 00) ──────────────────
import math

import numpy as np


def _dedup(ranked_ids):
    """Quita duplicados conservando la primera aparición (un doc repetido NO puntúa dos veces)."""
    seen, out = set(), []
    for d in ranked_ids:
        if d not in seen:
            seen.add(d)
            out.append(d)
    return out


def dcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """DCG con ganancia lineal (como trec_eval / pytrec_eval, que usa BEIR/MTEB) y descuento 1/log2(rango+1)."""
    ranked = _dedup(ranked_ids)[:k]
    return sum(rels.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranked) if rels.get(d, 0) > 0)


def ndcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """nDCG@k. rels: doc_id -> relevancia (>0 relevante). Sin relevantes -> 0.0 (evaluate() las excluye)."""
    ideal = sorted((r for r in rels.values() if r > 0), reverse=True)[:k]
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
    return 0.0 if idcg == 0 else dcg_at_k(ranked_ids, rels, k) / idcg


def recall_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """Fracción de los documentos relevantes que aparecen en el top-k."""
    relevant = {d for d, r in rels.items() if r > 0}
    if not relevant:
        return 0.0
    return len(relevant & set(_dedup(ranked_ids)[:k])) / len(relevant)


def mrr_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """1/rango del primer relevante dentro del top-k (0 si no hay ninguno)."""
    for i, d in enumerate(_dedup(ranked_ids)[:k]):
        if rels.get(d, 0) > 0:
            return 1.0 / (i + 1)
    return 0.0


def rank_from_scores(scores, doc_ids, k: int | None = None):
    """Ordena por score desc. Empates -> orden original del corpus (determinista). Los empates son una fuente
    silenciosa de varianza: con scores iguales, el orden de entrada decidiría el resultado."""
    scores = np.asarray(scores, dtype=float)
    order = np.lexsort((np.arange(len(scores)), -scores))
    order = order[:k] if k else order
    return [doc_ids[i] for i in order]


def evaluate(run: dict, qrels: dict, k: int = 10) -> dict:
    """run: {qid: [doc_id ordenados]}. Promedia solo sobre queries con algún relevante (convención trec_eval)."""
    n, tot = 0, {"ndcg": 0.0, "recall": 0.0, "mrr": 0.0}
    for qid, rels in qrels.items():
        if not any(r > 0 for r in rels.values()):
            continue
        ranked = run.get(qid, [])
        tot["ndcg"] += ndcg_at_k(ranked, rels, k)
        tot["recall"] += recall_at_k(ranked, rels, k)
        tot["mrr"] += mrr_at_k(ranked, rels, k)
        n += 1
    out = {m: (v / n if n else 0.0) for m, v in tot.items()}
    out.update(n_queries=n, n_skipped=len(qrels) - n)
    return out
WORD_RE = re.compile(r"\w+(?:['’]\w+)?|[^\w\s]", re.UNICODE)
def palabras(text: str): return WORD_RE.findall(text)

def metricas_tokenizacion(encode, textos) -> dict:
    """encode: str -> list[str|int]. Devuelve fertilidad (tokens/palabra), bytes/token y caracteres/token."""
    n_tok = n_pal = n_byte = n_chr = 0
    for t in textos:
        n_tok += len(encode(t)); n_pal += len(palabras(t)); n_byte += len(t.encode("utf-8")); n_chr += len(t)
    return {"fertilidad": n_tok / max(n_pal, 1), "bytes_por_token": n_byte / max(n_tok, 1), "chars_por_token": n_chr / max(n_tok, 1), "tokens": n_tok}

In [ ]:
import tiktoken
TIK = {}
for nombre in ["r50k_base", "cl100k_base", "o200k_base"]:
    try:
        TIK[nombre] = tiktoken.get_encoding(nombre)     # 1ª vez: descarga de openaipublic.blob.core.windows.net
    except Exception as e:
        print(f"(no pude cargar {nombre}: {type(e).__name__}; tras un proxy usa TIKTOKEN_CACHE_DIR). Se omitirá en los análisis.")
print({n: e.n_vocab for n, e in TIK.items()})

In [ ]:
import pandas as pd
from tokenizers import Tokenizer, models, pre_tokenizers, normalizers, trainers, decoders, Regex
from tokenizers.pre_tokenizers import ByteLevel
rng = random.Random(SEED); train_docs, test_docs = [], []
for lang in ["es", "en", "code"]:
    pool = [d for d in docs if d["lang"] == lang]; rng.shuffle(pool); cut = int(0.8 * len(pool))
    train_docs += pool[:cut]; test_docs += pool[cut:]
test_by_lang = {l: [d["text"] for d in test_docs if d["lang"] == l] for l in ["es", "en", "code"]}
print({l: len(v) for l, v in test_by_lang.items()}, "docs de test | train:", len(train_docs))

## 3. Mapa de madurez

Estado del arte a **oct-2026**; las cifras de vocabulario cambian con cada versión de modelo.

| Técnica / decisión | Madurez | Quién la usa (con respaldo) | Fuente |
|---|---|---|---|
| Vocabularios de 100–260 k (frente a 32 k) | 🟢 | cl100k ≈ 100 k y o200k ≈ 200 k (medido abajo); Llama 3 = 128 256; Qwen2 = 151 643 + 3; Gemma ≈ 256 k | tiktoken; Land & Bartolo 2024; Qwen2 report; Gemma 3 report |
| Dígitos tratados aparte (sueltos o en grupos) | 🟢 | Llama 1/2 y Gemma 3: dígitos separados; GPT-3.5/4: grupos de hasta 3 dígitos | Touvron 2023; Gemma 3; Singh & Strouse 2024 |
| Tokenizador específico con código/indentación (espacios agrupados) | 🟢 | cl100k/o200k agrupan espacios; GPT-2 no (medido abajo) | Dagan et al. 2024; medición propia |
| Vocabulario óptimo según presupuesto de cómputo (leyes de escala con V) | 🟡 | Resultados de NeurIPS 2024 a ≤ 3 B parámetros; extrapolación a modelos grandes no contrastada | Tao et al. 2024 |
| Equidad multilingüe del tokenizer (paridad) | 🟡 | Medida y reportada (p. ej. Tekken de Mistral frente al SentencePiece anterior); sin estándar común | Petrov et al. 2023; Ahia et al. 2023; docs de Mistral |
| Tokens que cruzan palabras (SuperBPE) | 🟡 / 🔴 | Resultados a 8 B, solo inglés; adopción en producción no verificada | Liu et al. 2025 (COLM) |
| Chat templates con tokens de control | 🟢 | Estándar de facto (ChatML, plantillas por modelo en `tokenizer_config`) | Docs de HF `transformers` |

## 4. Intuición primero: el vocabulario es un presupuesto

Cada token del vocabulario cuesta **parámetros** (una fila de $d$ números en la matriz de embeddings y otra en la de salida), **cómputo** (el softmax final es $O(V\cdot d)$ por token generado) y **datos** (cada fila necesita ver suficientes ejemplos para entrenarse; los tokens raros quedan *sub-entrenados*, módulo 05).
A cambio, un vocabulario mayor **acorta las secuencias**: menos tokens para el mismo texto = menos cómputo de atención ($O(n^2)$), más texto en la ventana de contexto y menos pasos de decodificación.

```
 V pequeño ─────────────────────────────────────────────► V grande
 + embeddings baratos            - secuencias largas      + secuencias cortas        - matriz grande, softmax caro
 + cada token muy entrenado      - contexto "efectivo"↓   - tokens raros sub-entrenados
 + robusto a typos               - más pasos al generar   - más sensible a datos de tokenizer≠modelo
```
> 💡 **Analogía de compresión.** Es el compromiso clásico *tamaño del diccionario ↔ longitud del mensaje* de LZ/Huffman: el diccionario también se transmite (aquí: se entrena y se guarda en el modelo).
> **Analogía recsys.** Equivale al tamaño de la tabla de embeddings de ítems: ítems de cola larga con pocos ejemplos tienen embeddings pobres; se reparten *hash buckets* o se agrupan.

> ### 🎨 GEMINI-IMAGEN · ID: `M04-IMG-01` · ⚠️ REEMPLAZAR ESTA CELDA
> **El vocabulario como presupuesto: qué se gana y qué se paga**
>
> **Qué muestra:** Balanza conceptual entre el tamaño del vocabulario V y sus costes/beneficios (embeddings, softmax, longitud de secuencia, tokens sub-entrenados).
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Esquema con el eje de tamaño del vocabulario V y dos columnas: lo que se gana (secuencias más cortas, más contexto) y lo que se paga (parámetros, softmax, tokens sub-entrenados), con una curva en U cuyo mínimo crece con el cómputo.
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, paleta sobria (azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280), tipografía sans-serif legible, flechas claras, sin ilustraciones decorativas ni dibujos de balanzas realistas. Formato 16:9.
> 
> Título arriba: "El vocabulario es un presupuesto".
> 
> Composición: una barra horizontal central con una flecha de doble punta; el extremo izquierdo etiquetado "V pequeño (p. ej. 32 k)" y el derecho "V grande (p. ej. 256 k)". Debajo de la barra, dos columnas de cajas.
> 
> Columna izquierda (borde azul), cabecera "Lo que se GANA al crecer V": cuatro cajas apiladas: "Secuencias más cortas (menos tokens por texto)", "Más texto cabe en el contexto", "Menos pasos al generar", "Mejor cobertura de idiomas y de código".
> Columna derecha (borde naranja), cabecera "Lo que se PAGA al crecer V": cuatro cajas apiladas: "Más parámetros: V × d en la matriz de entrada (y otra de salida si no hay weight tying)", "Softmax más caro: O(V · d) por token", "Tokens raros con pocos ejemplos: sub-entrenados", "Más superficie de ataque: glitch tokens".
> 
> Abajo, una franja gris con una curva en forma de U (sin números en los ejes) etiquetada en el eje X "Tamaño del vocabulario V" y en el eje Y "Pérdida a presupuesto de cómputo fijo", con un punto verde en el mínimo y el texto "óptimo: crece con el cómputo (Tao et al., 2024)".
> Evita: números inventados, texto ilegible, flechas ambiguas, iconos decorativos, sombras, degradados.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M04-IMG-01.png` en la carpeta del módulo y reemplaza esta celda por: `![Esquema con el eje de tamaño del vocabulario V y dos columnas: lo que se gana (secuencias más cortas, más contexto) y lo que se paga (parámetros, softmax, tokens sub-entrenados), con una curva en U cuyo mínimo crece con el cómputo.](assets/M04-IMG-01.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> V pequeño (~32k) <────────────────────────────> V grande (~256k)
> SE GANA al crecer V                      SE PAGA al crecer V
>  - secuencias más cortas                  - parámetros: V x d (x2 sin weight tying)
>  - más texto en el contexto               - softmax O(V·d) por token
>  - menos pasos al generar                 - tokens raros sub-entrenados
>  - mejor cobertura idiomas/código         - más glitch tokens
> pérdida a cómputo fijo: curva en U, mínimo en V* (crece con el cómputo)
> ```

> 👀 **Qué debes observar.** No existe un «mejor V» absoluto: es el mínimo de una curva en U que **se desplaza a la derecha cuando aumenta el presupuesto de cómputo** (resultado que formalizan Tao et al., 2024). Por eso los modelos grandes y recientes usan vocabularios mayores que los de 2020.

## 5. 📐 Teoría: los números del presupuesto

Sea $d$ la dimensión del modelo, $L$ el nº de capas y $V$ el tamaño del vocabulario. Aproximaciones habituales (Transformer denso estándar, FFN $4d$; las variantes GQA/SwiGLU cambian constantes pero no la conclusión):

$$
N_{\text{no-emb}}\approx 12\,L\,d^{2},\qquad N_{\text{emb}}=V\,d\ \ (\text{con weight tying}),\qquad N_{\text{emb}}=2\,V\,d\ \ (\text{sin tying}).
$$

- **Coste de salida por token generado:** $\approx 2\,V\,d$ FLOPs (multiplicar el estado final por la matriz $V\times d$ y luego softmax).
- **Longitud de secuencia:** para un texto de $B$ bytes, $n=B/\beta(V)$ con $\beta$ = bytes por token, creciente y cóncava en $V$ (lo medimos en el módulo 03: rendimientos decrecientes).
- **Coste de atención** $\propto n^2$ y de FFN/proyecciones $\propto n$: reducir $n$ un 20 % ahorra ≈ 20 % en la parte lineal y ≈ 36 % en la cuadrática.

### Leyes de escala con el vocabulario (Tao et al., 2024)
Entrenaron modelos de 33 M a 3 B de parámetros variando $V$ y observaron que **el vocabulario óptimo depende del presupuesto de cómputo**: los parámetros del vocabulario deben crecer, pero **más despacio** que los no-vocabulario;
predicen que el óptimo para Llama2-70B habría sido ≥ 216 K (frente a 32 K reales) y, a igual cómputo, pasar de 32 K a 43 K mejoró ARC-Challenge de 29.1 a 32.0 (resultados de los autores a esa escala; 🟡). Para comparar vocabularios distintos usan una pérdida **normalizada por caracteres/bytes**, no por token (si no, el vocabulario mayor «parece» peor porque predice tokens más difíciles).

> ⚠️ **Por qué no comparar perplexidad por token entre tokenizers.** Un tokenizer con tokens más largos predice *menos* tokens, cada uno con más información; su pérdida por token es mayor aunque el modelo sea igual de bueno. Compara **bits por byte** (o por carácter).

In [ ]:
# Fracción de parámetros que se va al vocabulario, para un Transformer "genérico" (aprox. 12·L·d² sin embeddings)
configs = {"A": (24, 1536), "B": (28, 1792), "C": (32, 4096), "D": (40, 5120), "E": (80, 8192)}   # (capas, d_model) genéricas
Vs = np.array([32_000, 50_257, 100_277, 128_256, 200_019, 256_000])
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.2))
for (nombre, (L, d)), c_ in zip(configs.items(), plt.cm.viridis(np.linspace(0.05, 0.9, len(configs)))):
    n_no = 12 * L * d ** 2
    ax[0].plot(Vs / 1e3, 100 * (2 * Vs * d) / (n_no + 2 * Vs * d), "o-", color=c_, label=f"L={L}, d={d} ({n_no/1e9:.1f} B sin emb.)")
ax[0].set_xlabel("Tamaño del vocabulario V (miles)"); ax[0].set_ylabel("% de parámetros en embeddings (sin weight tying)"); ax[0].set_title("El vocabulario pesa MUCHO en modelos pequeños"); ax[0].legend(fontsize=7)
# Ahorro de cómputo al acortar la secuencia: lineal vs cuadrático
red = np.linspace(0, 0.5, 50)
ax[1].plot(100 * red, 100 * red, label="partes lineales (FFN, proyecciones)", c=COL["azul"], lw=2)
ax[1].plot(100 * red, 100 * (1 - (1 - red) ** 2), label="atención (∝ n²)", c=COL["naranja"], lw=2)
ax[1].set_xlabel("Reducción de tokens por texto (%)"); ax[1].set_ylabel("Ahorro de FLOPs (%)"); ax[1].set_title("Acortar la secuencia ahorra más en la atención"); ax[1].legend()
plt.tight_layout(); plt.show()
print("Ejemplo d=4096, L=32 (≈6.4 B no-emb): V=32k ->", f"{2*32000*4096/1e9:.2f} B en embeddings;", " V=256k ->", f"{2*256000*4096/1e9:.2f} B")

> 👀 **Qué debes observar.** **Izquierda:** en un modelo de ≈ 0.7–1 B parámetros sin embeddings, pasar de 32 k a 256 k tokens mete **de ≈ 10–13 % a ≈ 46–54 %** de los parámetros totales en las matrices de embedding (sin weight tying): los modelos pequeños con vocabularios enormes son en buena parte «tablas de embeddings».
> En un modelo de ≈ 64 B sin embeddings el mismo cambio cuesta de < 1 % a ≈ 6 %. **Derecha:** un 20 % menos de tokens ahorra ≈ 20 % en lo lineal y ≈ 36 % en la atención: la compresión del tokenizer es un multiplicador de cómputo.

## 6. Vocabularios de modelos reales (verificado, fechado)

Cifras **a oct-2026** y solo las que he podido contrastar con una fuente primaria o con una medición propia. Los modelos lanzados después (o cuyos tokenizers no pude abrir) **no** están en la tabla: verifica con `AutoTokenizer.from_pretrained(...).vocab_size`/`len(tokenizer)` (¡las dos cifras pueden diferir por tokens añadidos!).

| Modelo / tokenizer | Algoritmo | Vocabulario | Fuente |
|---|---|---|---|
| GPT-2 (`r50k_base`) | byte-level BPE + regex | 50 257 | medido con `tiktoken` (arriba) |
| GPT-3.5/GPT-4 (`cl100k_base`) | byte-level BPE + regex | 100 277 (incl. especiales) | medido con `tiktoken` |
| GPT-4o (`o200k_base`) | byte-level BPE + regex | 200 019 (incl. especiales) | medido con `tiktoken` |
| Llama 2 | BPE vía SentencePiece | 32 000 | Land & Bartolo 2024 (tabla 1) |
| Llama 3 | tiktoken (100 K) + 28 K no ingleses | 128 256 | Llama 3 report; Land & Bartolo 2024 |
| Qwen2 | byte-level BPE | 151 643 regulares + 3 de control | Qwen2 technical report |
| Gemma (1/2) | SentencePiece | 256 000 | Land & Bartolo 2024 (tabla 1) |
| Gemma 3 | SentencePiece (dígitos separados, bytes) | ≈ 262 k (texto) / 256 k (pie de tabla): el informe es inconsistente | Gemma 3 report |
| Mistral NeMo y posteriores (**Tekken**) | BPE basado en tiktoken | ≈ 130 k + ≈ 1 k tokens de control | Docs de Mistral (tokenization guide) |
| Command R (Cohere) | BPE | 255 029 | Land & Bartolo 2024 (tabla 1) |

> 🧠 **Patrón:** de 32 k (2023) a 100–256 k (2024–2026). Las dos razones, ambas medibles aquí: *(i)* mejor compresión multilingüe y de código, *(ii)* el vocabulario pesa poco en modelos grandes (gráfico anterior).

In [ ]:
# Registro de tokenizers a comparar: nombre -> función texto -> lista de tokens (ids o strings). Solo cuenta len().
def entrena_bpe_bytes(textos, vocab_size: int) -> Tokenizer:
    """Byte-level BPE (regex GPT-2) entrenado aquí: nuestro 'vocabulario propio'."""
    PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    t = Tokenizer(models.BPE())
    t.pre_tokenizer = pre_tokenizers.Sequence([pre_tokenizers.Split(Regex(PAT), behavior="isolated"), ByteLevel(add_prefix_space=False, use_regex=False)])
    t.decoder = decoders.ByteLevel()
    t.train_from_iterator(textos, trainers.BpeTrainer(vocab_size=vocab_size, initial_alphabet=ByteLevel.alphabet(), special_tokens=["<|endoftext|>"]))
    return t

V_PROPIO = 4000 if FAST_DEV_RUN else 32000
bpe_propio = entrena_bpe_bytes([d["text"] for d in train_docs], V_PROPIO)
REG = {f"tiktoken {n.replace('_base','')}": (lambda s, e=e: e.encode_ordinary(s)) for n, e in TIK.items()}
REG[f"BPE propio ({V_PROPIO//1000}k, ES/EN/código)"] = lambda s: bpe_propio.encode(s).ids

# Tokenizers de modelos reales vía HF (cuando hay red). IDs sin verificar desde este entorno: la celda salta los que fallen.
MODELOS_HF = {"GPT-2": "gpt2", "Qwen2.5": "Qwen/Qwen2.5-0.5B", "XLM-R": "FacebookAI/xlm-roberta-base",
              "Mistral-NeMo (Tekken)": "mistralai/Mistral-Nemo-Instruct-2407", "Llama-3.2": "unsloth/Llama-3.2-1B"}
cargados = []
if hay_red("https://huggingface.co"):
    from transformers import AutoTokenizer
    for nombre, repo in MODELOS_HF.items():
        try:
            tk = AutoTokenizer.from_pretrained(repo)
            REG[f"{nombre} [HF]"] = (lambda s, tk=tk: tk.encode(s, add_special_tokens=False)); cargados.append(f"{nombre}: len(tokenizer)={len(tk)}")
        except Exception as e:
            print(f"   (omitido {nombre}: {type(e).__name__})")
    print("tokenizers HF cargados:", cargados)
else:
    print("Sin acceso a HF: se analizan los tokenizers disponibles localmente (tiktoken + el nuestro). Con red, esta celda añade tokenizers de modelos reales.")
print(list(REG))

## 7. Multilingüe y «token tax»: medir la paridad

Un tokenizer entrenado mayoritariamente con inglés necesita **más tokens para decir lo mismo** en otros idiomas. Petrov et al. (2023) lo cuantifican sobre texto **paralelo** (la misma frase traducida) y encuentran diferencias de hasta **15×** entre idiomas
con los tokenizers de LLM que analizaron, que persisten incluso en los pensados para ser multilingües; Ahia et al. (2023) lo traducen en **precio**: los hablantes de muchos idiomas pagan más por la misma información en APIs comerciales y reciben peor resultado.

**Prima de tokenización** (*premium*) de un idioma $\ell$ frente al inglés, sobre $n$ frases paralelas:

$$
\text{premium}(\ell)=\frac{\sum_{i=1}^{n}|\text{tok}(s_i^{\ell})|}{\sum_{i=1}^{n}|\text{tok}(s_i^{\text{en}})|}\qquad(\text{1.0 = paridad con el inglés}).
$$

Y la métrica complementaria **bytes por token** $\beta_\ell=\sum |s^\ell|_{\text{bytes}}/\sum|\text{tok}|$. Un tokenizer «justo» no tiene $\beta$ igual en todos los idiomas (los scripts ocupan distinto nº de bytes), pero sí debería tener un *premium* cercano a 1.

> ⚠️ **Honestidad de datos.** Con FLORES+ (≈ 1000 frases/idioma) las cifras son fiables. Con el mini-conjunto de 16 frases escrito a mano son **orientativas**; ordena los idiomas, pero no cites decimales.

In [ ]:
def tabla_paridad(par: dict, registro: dict, langs=None) -> pd.DataFrame:
    langs = langs or [l for l in par["langs"]]
    filas = []
    for nombre, enc in registro.items():
        tot = {l: sum(len(enc(s)) for s in par["langs"][l]) for l in langs}
        byt = {l: sum(len(s.encode("utf-8")) for s in par["langs"][l]) for l in langs}
        for l in langs:
            filas.append({"tokenizer": nombre, "idioma": l, "tokens": tot[l], "premium": tot[l] / tot["en"], "bytes_por_token": byt[l] / tot[l]})
    return pd.DataFrame(filas)

dp = tabla_paridad(par, REG)
prem = dp.pivot_table(index="tokenizer", columns="idioma", values="premium")[list(par["langs"])]
bpt = dp.pivot_table(index="tokenizer", columns="idioma", values="bytes_por_token")[list(par["langs"])]
print(prem.round(2).to_string())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2), gridspec_kw={"width_ratios": [1.25, 1]})
im = ax[0].imshow(prem.values, cmap="YlOrRd", vmin=1, vmax=max(3, prem.values.max()))
ax[0].set_xticks(range(prem.shape[1])); ax[0].set_xticklabels(prem.columns); ax[0].set_yticks(range(prem.shape[0])); ax[0].set_yticklabels(prem.index, fontsize=8); ax[0].grid(False)
for i in range(prem.shape[0]):
    for j in range(prem.shape[1]): ax[0].text(j, i, f"{prem.values[i, j]:.1f}", ha="center", va="center", fontsize=8, color="white" if prem.values[i, j] > 2.4 else "black")
ax[0].set_title("Prima de tokenización frente al inglés (1.0 = paridad)"); plt.colorbar(im, ax=ax[0], fraction=0.03)
sel = [n for n in prem.index if "o200k" in n or "cl100k" in n or "propio" in n]
for n, c_ in zip(sel, [COL["azul"], COL["verde"], COL["naranja"]]): ax[1].plot(bpt.columns, bpt.loc[n].values, "o-", label=n, color=c_, lw=2)
ax[1].set_ylabel("Bytes de texto por token"); ax[1].set_title("Cuánto texto cubre cada token, por idioma"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar** (mini-conjunto de 16 frases; orientativo).
> **(1)** Con `cl100k` las escrituras alejadas del latín pagan mucho más que el inglés (hindi ≈ 4.9×, árabe ≈ 3×, ruso ≈ 2.1×) y los idiomas latinos algo menos pero **no** 1.0 (español ≈ 1.3×). Con `r50k` (GPT-2) es peor aún (hindi ≈ 7×, ruso ≈ 5.5×).
> **(2)** `o200k` comprime la brecha: hindi ≈ 1.6×, árabe ≈ 1.2×, ruso ≈ 1.25×; pero el **japonés sigue siendo el más caro (≈ 1.85×)**. Ampliar el vocabulario hacia otros idiomas es exactamente lo que cuestan los 100 k tokens extra.
> **(3)** **Nuestro BPE propio** (solo ES/EN/código) es *favorable* al español (≈ 1.2×) pero muy injusto con ru/hi/ja/ko/ar (≈ 4–8×): casi un token por byte. Si el producto atenderá esos idiomas, hay que incluirlos en el corpus del tokenizer.

> ### 🎨 GEMINI-IMAGEN · ID: `M04-IMG-02` · ⚠️ REEMPLAZAR ESTA CELDA
> **El impuesto por idioma: la misma frase, distinto número de tokens**
>
> **Qué muestra:** Ilustra de forma cualitativa que una misma frase traducida se parte en más o menos tokens según el idioma y que eso se traduce en coste y contexto.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Cuatro filas con la misma frase en inglés, español, ruso e hindi y un número creciente de fichas (tokens) ilustrativas, con la etiqueta de coste relativo y tres consecuencias: dinero, contexto y latencia.
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, paleta sobria (azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280), tipografía sans-serif legible, sin ilustraciones decorativas. Formato 16:9.
> 
> Título arriba: "Token tax: la misma frase, distinto coste".
> 
> Cuatro filas horizontales. Cada fila tiene a la izquierda el nombre del idioma y la frase, y a la derecha una cadena de "fichas" rectangulares (chips) que representan los tokens (los chips NO llevan texto, solo son rectángulos; su número es ilustrativo).
> Fila 1, "Inglés": "The model was trained on many languages." con 8 chips azules.
> Fila 2, "Español": "El modelo se entrenó con muchos idiomas." con 11 chips verdes.
> Fila 3, "Ruso": "Модель обучена на многих языках." con 17 chips naranjas.
> Fila 4, "Hindi": "मॉडल को कई भाषाओं पर प्रशिक्षित किया गया था।" con 34 chips grises.
> Al final de cada fila, una barra horizontal proporcional al nº de chips y una etiqueta "×1.0", "×1.4", "×2.1", "×4.3" respectivamente (marcadas con el texto "valores ilustrativos").
> Debajo, tres cajas pequeñas en una fila: "$ más caro por la misma información", "menos contexto útil en la ventana", "más latencia al generar".
> Pie en letra pequeña gris: "Cifras ilustrativas; mide tu tokenizer con texto paralelo (FLORES)".
> Evita: presentar los números como mediciones reales, texto ilegible, iconos decorativos, banderas, sombras, degradados.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M04-IMG-02.png` en la carpeta del módulo y reemplaza esta celda por: `![Cuatro filas con la misma frase en inglés, español, ruso e hindi y un número creciente de fichas (tokens) ilustrativas, con la etiqueta de coste relativo y tres consecuencias: dinero, contexto y latencia.](assets/M04-IMG-02.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> Inglés  The model was trained on many languages.     [][][][][][][][]                     x1.0
> Español El modelo se entrenó con muchos idiomas.     [][][][][][][][][][]                 x1.4
> Ruso    Модель обучена на многих языках.             [][][][][][][][][][][][][][][][][]   x2.1
> Hindi   मॉडल को कई भाषाओं पर प्रशिक्षित ...          [][][]...(34 fichas)                  x4.3   (ilustrativo)
> consecuencias: $ más caro | menos contexto útil | más latencia
> ```

> 👀 **Qué debes observar.** Lo que importa no es la cifra exacta (ilustrativa aquí) sino la **cadena causal**: más tokens por la misma información → más dinero (se factura por token), menos documentos en el contexto y más pasos de decodificación. Compáralo con tu tabla de paridad real de arriba.

## 8. Del token tax al dinero y al contexto útil

Si una API factura a $p$ dólares por millón de tokens (**valor que debes leer en la tarifa vigente de tu proveedor: aquí es un parámetro ilustrativo**), el coste de procesar la **misma información** en el idioma $\ell$ es

$$
\text{coste}(\ell)=\text{premium}(\ell)\cdot c_{\text{en}},\qquad c_{\text{en}}=\frac{p}{10^{6}}\cdot\overline{\text{tok}}_{\text{en}}\cdot n_{\text{frases}}.
$$

Y, para una ventana de contexto de $W$ tokens, cabe $W/(\overline{\text{tok}}_{\ell})$ frases del idioma $\ell$ (**contexto efectivo**). Con lo medido arriba:

In [ ]:
PRECIO_USD_POR_MTOK = 1.0     # ILUSTRATIVO: sustituye por la tarifa vigente de tu proveedor (varía por modelo y fecha)
VENTANA = 128_000             # tokens de contexto (parámetro)
N_FRASES_MES = 50_000_000     # volumen de ejemplo: frases procesadas al mes

ref = "tiktoken o200k" if "tiktoken o200k" in prem.index else prem.index[0]
sub = dp[dp.tokenizer == ref].set_index("idioma"); n_par = par["n"]
tok_por_frase = sub["tokens"] / n_par
coste = tok_por_frase * N_FRASES_MES / 1e6 * PRECIO_USD_POR_MTOK
cabe = VENTANA / tok_por_frase
res = pd.DataFrame({"tokens/frase": tok_por_frase.round(1), "premium": sub["premium"].round(2), "coste_mes_USD": coste.round(0), "frases_en_ventana": cabe.round(0)})
print(f"Tokenizer: {ref} | precio ilustrativo {PRECIO_USD_POR_MTOK} USD/Mtok | volumen {N_FRASES_MES:,} frases/mes | ventana {VENTANA:,}")
print(res.to_string())
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4))
orden = res["premium"].sort_values().index
ax[0].bar(orden, res.loc[orden, "coste_mes_USD"] / 1e3, color=[COL["azul"] if l == "en" else COL["naranja"] for l in orden]); ax[0].set_ylabel("Coste mensual (miles de USD, ilustrativo)"); ax[0].set_title(f"Misma información, distinto coste ({ref})")
ax[1].bar(orden, res.loc[orden, "frases_en_ventana"] / 1e3, color=[COL["azul"] if l == "en" else COL["verde"] for l in orden]); ax[1].set_ylabel("Frases que caben en la ventana (miles)"); ax[1].set_title(f"Contexto efectivo con W = {VENTANA:,}")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** Los dos gráficos son **la misma información leída de dos formas**: el idioma con mayor prima paga más y, a la vez, mete menos texto en la ventana. Con el precio y el volumen ilustrativos de la celda el recargo del japonés es ≈ +85 % sobre el inglés (≈ 500 USD/mes más); con tarifas reales y volúmenes de producto es una **decisión de negocio** (y de equidad) que se toma al diseñar el tokenizer, no después.

## 9. Números, espacios e indentación

**Números.** Si «1234567» se parte como `123|456|7` (grupos de 3 de izquierda a derecha, estilo GPT-3.5/4) las columnas de unidades de dos sumandos **no se alinean** con las del resultado.
Singh & Strouse (2024) mostraron en GPT-3.5/GPT-4 (modelos de marzo de 2023) que tokenizar **de derecha a izquierda** (forzado separando miles con comas) sube la precisión de la suma de 75.6 % a 97.8 % en GPT-3.5 (8-shot, sumandos de 7–9 dígitos), con un efecto algo menor pero presente en GPT-4: el error venía de la **alineación de tokens**, no de «no saber sumar».
Llama 1/2 y Gemma 3 optan por **dígitos sueltos** (alineación perfecta, secuencias más largas). Es un diseño con trade-offs; la evidencia a escala actual es 🟡.

**Espacios.** GPT-2 usaba un token por espacio de indentación; cl100k/o200k tienen tokens para **runs de espacios**. Dagan et al. (2024) muestran que la regex de pre-tokenización y el tamaño cambian velocidad, contexto efectivo y rendimiento en código. Medimos ambos efectos.

In [ ]:
numeros = ["7", "42", "1234", "1234567", "3.14159", "20241025", "1,234,567"]
if TIK:
    for n, e in TIK.items():
        print(f"{n:>12}:", {x: [e.decode_single_token_bytes(i).decode() for i in e.encode_ordinary(x)] for x in ["1234567", "3.14159", "1,234,567"]})
    print("\npatrón de números cl100k:", [p for p in TIK["cl100k_base"]._pat_str.split("|") if "N" in p][:2])
    # tokens por número según longitud (n dígitos)
    longitudes = list(range(1, 16)); r_ = random.Random(SEED)
    cuenta = {n: [np.mean([len(e.encode_ordinary("".join(r_.choice("0123456789") for _ in range(L)))) for _ in range(80)]) for L in longitudes] for n, e in TIK.items()}
    cuenta["dígitos sueltos"] = longitudes
# indentación: tokens por línea de código real con n espacios de indentación
def costo_indent(enc, codigos):
    por_nivel = collections.defaultdict(list)
    for c in codigos:
        for ln in c.split("\n"):
            if ln.strip():
                sp = len(ln) - len(ln.lstrip(" "))
                por_nivel[sp].append(len(enc(ln)) - len(enc(ln.lstrip(" "))))
    return {k: float(np.mean(v)) for k, v in sorted(por_nivel.items()) if k in (0, 4, 8, 12, 16) and len(v) >= 5}
codigos = [d["text"] for d in docs if d["lang"] == "code"]
ind = {nombre: costo_indent(enc, codigos) for nombre, enc in REG.items() if "tiktoken" in nombre or "propio" in nombre}
print({k: {n: round(v, 2) for n, v in d.items()} for k, d in ind.items()})

In [ ]:
if TIK:
    fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.2))
    for (n, ys), c_ in zip(cuenta.items(), [COL["gris"], COL["azul"], COL["verde"], COL["rojo"]]): ax[0].plot(longitudes, ys, "o-", label=n, color=c_, lw=2, ms=4)
    ax[0].set_xlabel("Dígitos del número"); ax[0].set_ylabel("Tokens"); ax[0].set_title("Tokens por número aleatorio"); ax[0].legend(fontsize=8)
    niveles = [0, 4, 8, 12, 16]; w = 0.2
    for k, ((nombre, d), c_) in enumerate(zip(ind.items(), [COL["gris"], COL["azul"], COL["verde"], COL["naranja"]])):
        ax[1].bar(np.arange(len(niveles)) + (k - 1.5) * w, [d.get(n_, 0) for n_ in niveles], w, label=nombre, color=c_)
    ax[1].set_xticks(range(len(niveles))); ax[1].set_xticklabels([f"{n_} espacios" for n_ in niveles]); ax[1].set_ylabel("Tokens extra por la indentación (media por línea)")
    ax[1].set_title("Coste de la indentación en código Python real"); ax[1].legend(fontsize=7)
    plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** **Izquierda:** con `cl100k`/`o200k` un número crece **a saltos de 3 dígitos** (≈ 1 token por cada 3 cifras); GPT-2 (`r50k`) usa mezclas irregulares; solo «dígitos sueltos» es lineal y alineado. Cada diseño elige entre **longitud** y **alineación aritmética**.
> **Derecha:** en GPT-2 la indentación cuesta ≈ **un token por espacio** (16 espacios ≈ 16 tokens), mientras cl100k/o200k la colapsan a ≈ 1 token: la mitad del «peso» de un archivo de código con indentación profunda desaparece por una decisión de regex/vocabulario (Dagan et al., 2024).

## 10. Tokens especiales y *chat templates*

Los **tokens especiales** (`<|endoftext|>`, `<s>`, `<|im_start|>`…) no se aprenden por frecuencia: se **reservan** en el vocabulario y se insertan por código. Hay tres familias:

1. **Estructura de secuencia:** BOS/EOS/PAD/UNK/SEP (`[CLS]`, `[SEP]` en BERT).
2. **Roles y turnos de chat:** `<|im_start|>user … <|im_end|>` (ChatML), `<|start_header_id|>` en Llama 3, `[INST]` en Mistral v1–v3… cada familia de modelos tiene **su** plantilla.
3. **Capacidades:** *fill-in-the-middle* (`<|fim_prefix|>…`), herramientas, imágenes, «pensamiento».

Un **chat template** es la función que convierte `[{"role": ..., "content": ...}, ...]` en la cadena exacta (con sus tokens especiales) con la que el modelo fue *post-entrenado*. En HF vive en `tokenizer.chat_template` (Jinja) y se aplica con `tokenizer.apply_chat_template`.
**Regla de oro:** la plantilla en servicio debe ser **idéntica** a la de entrenamiento; un espacio o un salto de línea de más degrada resultados de forma silenciosa.

> ### 🎨 GEMINI-IMAGEN · ID: `M04-IMG-03` · ⚠️ REEMPLAZAR ESTA CELDA
> **Anatomía de una conversación con chat template (ChatML)**
>
> **Qué muestra:** Muestra cómo un diálogo de 3 mensajes se convierte en una secuencia con tokens especiales y qué tokens entran en la pérdida del post-entrenamiento.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Tres mensajes (system, user, assistant) convertidos con apply_chat_template en una cadena ChatML con tokens especiales, y la misma secuencia con los tokens del turno assistant marcados como parte de la pérdida.
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, paleta sobria (azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280), tipografía sans-serif legible y monoespaciada para la cadena, flechas claras. Formato 16:9.
> 
> Título arriba: "De mensajes a tokens: el chat template".
> 
> Parte superior (izquierda): una lista de 3 mensajes en cajas apiladas, cada una con su rol en una etiqueta de color: [system] "Eres un asistente de Políglota Labs." (gris), [user] "¿Qué es un token?" (azul), [assistant] "Un trozo de texto que el modelo usa como unidad." (verde).
> Flecha ancha hacia la derecha etiquetada "apply_chat_template".
> Parte superior (derecha): la cadena resultante en una caja monoespaciada, con los tokens especiales resaltados en naranja, exactamente así (cada línea en su renglón):
> <|im_start|>system
> Eres un asistente de Políglota Labs.<|im_end|>
> <|im_start|>user
> ¿Qué es un token?<|im_end|>
> <|im_start|>assistant
> Un trozo de texto que el modelo usa como unidad.<|im_end|>
> 
> Parte inferior: la misma secuencia dibujada como una fila de chips de izquierda a derecha agrupados en tres bloques (system, user, assistant). Debajo de cada chip una línea indica "pérdida": los chips del bloque assistant (incluido <|im_end|>) tienen una marca verde "entra en la pérdida (label)"; los de system y user tienen una marca gris "enmascarado (-100)".
> Leyenda abajo a la derecha: naranja = token especial, verde = parte entrenada, gris = contexto.
> Nota pequeña: "La plantilla de servicio debe ser idéntica a la de entrenamiento".
> Evita: texto ilegible, cadenas distintas a las indicadas, flechas ambiguas, iconos decorativos, sombras, degradados.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M04-IMG-03.png` en la carpeta del módulo y reemplaza esta celda por: `![Tres mensajes (system, user, assistant) convertidos con apply_chat_template en una cadena ChatML con tokens especiales, y la misma secuencia con los tokens del turno assistant marcados como parte de la pérdida.](assets/M04-IMG-03.png)`
>
> **Mientras tanto (placeholder):**
> ```text
>  [system] Eres un asistente...   --apply_chat_template-->
>  [user]   ¿Qué es un token?          <|im_start|>system
> Eres un asistente...<|im_end|>
> 
>  [assistant] Un trozo de texto...     <|im_start|>user
> ¿Qué es un token?<|im_end|>
> 
>                                        <|im_start|>assistant
> Un trozo de texto...<|im_end|>
>  pérdida:  system/user = enmascarado (-100) | assistant (+ <|im_end|>) = entra en la pérdida
> ```

> 👀 **Qué debes observar.** (1) Los roles son **texto + tokens especiales**, no una API mágica. (2) En el post-entrenamiento solo se calcula la pérdida sobre las respuestas del asistente (y su token de cierre, para que aprenda a *parar*); el resto se enmascara. (3) Si en servicio olvidas el `<|im_start|>assistant\n` final, el modelo no sabe que le toca hablar.

In [ ]:
# 1) Ampliar un Encoding de tiktoken con tokens de chat (receta del README de tiktoken)
if TIK:
    cl = TIK["cl100k_base"]
    cl_im = tiktoken.Encoding(name="cl100k_im", pat_str=cl._pat_str, mergeable_ranks=cl._mergeable_ranks,
                              special_tokens={**cl._special_tokens, "<|im_start|>": 100264, "<|im_end|>": 100265})
    def chatml(mensajes, generar=True):
        s = "".join(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n" for m in mensajes)
        return s + ("<|im_start|>assistant\n" if generar else "")
    msgs = [{"role": "system", "content": "Eres un asistente de Políglota Labs."}, {"role": "user", "content": "¿Qué es un token?"}]
    txt = chatml(msgs); ids = cl_im.encode(txt, allowed_special="all")
    print(repr(txt)); print(len(ids), "tokens | especiales presentes:", [i for i in ids if i >= 100257])
    # máscara de pérdida: solo lo que viene DESPUÉS de '<|im_start|>assistant\n' hasta '<|im_end|>'
    completa = chatml(msgs + [{"role": "assistant", "content": "Un trozo de texto."}], generar=False)
    ids_c = cl_im.encode(completa, allowed_special="all"); inicio = len(cl_im.encode(chatml(msgs), allowed_special="all"))
    mask = [0] * inicio + [1] * (len(ids_c) - inicio - 1)  + [0]       # entrena respuesta + <|im_end|>; el '\n' final no
    print("tokens con pérdida:", [cl_im.decode([i]) for i, m in zip(ids_c, mask) if m])

In [ ]:
# 2) El mismo mecanismo con HF: chat_template Jinja en un tokenizer propio
from transformers import PreTrainedTokenizerFast
from tokenizers import processors
msgs = [{"role": "system", "content": "Eres un asistente de Políglota Labs."}, {"role": "user", "content": "¿Qué es un token?"}]
bpe_chat = entrena_bpe_bytes([d["text"] for d in train_docs] + ["<|im_start|> <|im_end|> system user assistant"], V_PROPIO)
bos_id = bpe_chat.token_to_id("<|endoftext|>")
bpe_chat.post_processor = processors.TemplateProcessing(single="<|endoftext|> $A", special_tokens=[("<|endoftext|>", bos_id)])   # BOS automático
hf = PreTrainedTokenizerFast(tokenizer_object=bpe_chat, bos_token="<|endoftext|>", eos_token="<|endoftext|>", pad_token="<|endoftext|>")
hf.add_special_tokens({"additional_special_tokens": ["<|im_start|>", "<|im_end|>"]})
hf.chat_template = ("{% for m in messages %}<|im_start|>{{ m['role'] }}\n{{ m['content'] }}<|im_end|>\n{% endfor %}"
                    "{% if add_generation_prompt %}<|im_start|>assistant\n{% endif %}")
salida = hf.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
print(repr(salida))
# Error clásico (doble BOS): la plantilla ya escribió el BOS en el texto y el tokenizer añade otro al re-tokenizar
plantilla_con_bos = "<|endoftext|>" + salida
con = hf(plantilla_con_bos)["input_ids"]; sin = hf(plantilla_con_bos, add_special_tokens=False)["input_ids"]
print("add_special_tokens=True  ->", con.count(bos_id), "BOS al inicio:", con[:3])
print("add_special_tokens=False ->", sin.count(bos_id), "BOS al inicio:", sin[:3])

**Errores típicos con plantillas** (los verás en producción): (1) **doble BOS**: la plantilla ya añade `<s>` y el tokenizer lo añade otra vez al tokenizar el texto resultante (`add_special_tokens=False` al re-tokenizar); (2) plantilla de *entrenamiento* ≠ de *servicio* (un `\n` de más);
(3) olvidar `add_generation_prompt=True`; (4) enmascarar mal la pérdida (entrenar también sobre el prompt del usuario); (5) **inyección de tokens de control** desde el texto del usuario (si el usuario escribe `<|im_end|>` y tu tokenizer lo convierte en el token real, puede cerrar el turno: módulo 05).

## 11. Tokens que cruzan palabras: SuperBPE

Toda la tubería de módulos 02–03 asume que **un token no cruza una palabra**. Pero «a lot of», «de la», «search engine» funcionan como unidades semánticas. **SuperBPE** (Liu et al., 2025; COLM 2025) añade un **currículo de dos etapas** al BPE:

1. **Etapa 1** (hasta un punto de transición $t$): BPE normal con pre-tokenización por espacios → aprende **subpalabras**.
2. **Etapa 2** (de $t$ a $T$): se **desactiva** la pre-tokenización por espacios y se sigue fusionando → aparecen **superpalabras** (tokens que cruzan espacios).

Resultados que reportan los autores (8 B de parámetros, vocabulario de 200 k, entrenados desde cero, **solo inglés**): hasta ≈ 33 % menos tokens para el mismo texto, +4.0 puntos de media en 30 tareas (+8.2 en MMLU) con ≈ 27 % menos cómputo de inferencia; pero **empeora LAMBADA** (77.0 → 70.6) y exige más CPU/memoria para entrenar el tokenizer, y requiere deduplicar para evitar superpalabras de *boilerplate*.
Estado: 🟡 (resultado de un grupo, escala 8 B, inglés); no he verificado adopción en producción.

Vamos a implementar la **versión mínima**: etapa 1 con `BpeTrainer` de HF (palabras), etapa 2 con fusiones sobre listas de tokens que **sí pueden cruzar espacios**, y comparamos con BPE estándar al **mismo tamaño total de vocabulario**.

> ### 🎬 GEMINI-VIDEO · ID: `M04-VID-01` · ⚠️ REEMPLAZAR ESTA CELDA
> **SuperBPE: de subpalabras a superpalabras**
>
> **Qué muestra:** Animación de las dos etapas de SuperBPE sobre la frase «en el caso de la ley» mostrando cómo, al desactivar la pre-tokenización por espacios, aparecen tokens que cruzan palabras.
> **Formato:** MP4 16:9, 1080p | Duración: 30 s | **Texto alternativo:** Animación de 30 s: la frase «en el caso de la ley» pasa de 7 a 5 tokens al apagar la pre-tokenización por espacios y fusionar a través de las palabras en la etapa 2 de SuperBPE.
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Animación explicativa 16:9, estilo diagrama técnico limpio: fondo blanco, paleta sobria (azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280), tipografía sans-serif y monoespaciada para los tokens, sin ilustraciones decorativas, sin voz en off, ritmo calmado. Texto en pantalla en español. Duración total 30 segundos.
> 
> Frase de ejemplo (usa exactamente esta): "en el caso de la ley". El símbolo ▁ marca el inicio de palabra.
> 
> Escena 1 (0–6 s): Título "SuperBPE: dos etapas". Se muestra la frase como una fila de chips grises, un chip por palabra: [▁en] [▁el] [▁caso] [▁de] [▁la] [▁ley]. Debajo, una barrera vertical roja (#D62728) entre cada par de chips con el texto "pre-tokenización por espacios: ningún token cruza la barrera".
> Escena 2 (6–14 s): Etiqueta "Etapa 1 (BPE normal, hasta el punto de transición t)". Los chips de palabras se parten en subpalabras con una animación suave: [▁en] [▁el] [▁ca][so] [▁de] [▁la] [▁ley] y luego vuelven a fusionarse parcialmente; aparece un contador "|V| crece hasta t".
> Escena 3 (14–20 s): Etiqueta "Transición: se APAGA la pre-tokenización por espacios". Las barreras rojas se desvanecen.
> Escena 4 (20–28 s): Etiqueta "Etapa 2 (de t a T)": los pares adyacentes más frecuentes se fusionan a través de los espacios, uno por uno, con una flecha naranja por cada fusión: [▁en][▁el] → [▁en▁el]; [▁de][▁la] → [▁de▁la]. Resultado final en una fila: [▁en▁el] [▁ca][so] [▁de▁la] [▁ley] (4 chips + 1 = 5 chips de 7 iniciales, mostrados como "7 → 5 tokens").
> Escena 5 (28–30 s): Texto final: "Menos tokens por texto. Límites: probado a 8 B, solo en inglés".
> Evita: números distintos a los indicados, texto ilegible, flechas que se crucen, elementos decorativos, transiciones rápidas, música o voz.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M04-VID-01.mp4` en la carpeta del módulo y reemplaza esta celda por: `<video src="assets/M04-VID-01.mp4" controls width="100%"></video>`
>
> **Mientras tanto (placeholder):**
> ```text
> Etapa 1 (palabras):   [▁en] | [▁el] | [▁ca][so] | [▁de] | [▁la] | [▁ley]      (7 tokens; | = barrera de espacios)
> Transición:           se apaga la pre-tokenización por espacios
> Etapa 2 (superpalabras): [▁en▁el] [▁ca][so] [▁de▁la] [▁ley]                    (5 tokens)
> ```

> 👀 **Qué debes observar.** La ganancia viene **solo de la etapa 2**, que fusiona pares *entre* palabras que ya son frecuentes (`▁de▁la`, `▁en▁el`). Es compresión pura; que además mejore el modelo (como reportan los autores) es una afirmación empírica aparte y todavía no replicada a gran escala.

In [ ]:
Counter = collections.Counter
def bpe_palabras(textos, V: int) -> Tokenizer:
    t = Tokenizer(models.BPE(unk_token="[UNK]")); t.normalizer = normalizers.NFKC()
    t.pre_tokenizer = pre_tokenizers.Metaspace(replacement="▁", prepend_scheme="always")
    t.train_from_iterator(textos, trainers.BpeTrainer(vocab_size=V, special_tokens=["[UNK]"], initial_alphabet=[chr(i) for i in range(32, 127)]))
    return t

def superbpe(textos, t_etapa1: int, T: int):
    """Etapa 1: BPE por palabras hasta t_etapa1. Etapa 2: fusiones entre tokens ADYACENTES aunque crucen espacios, hasta T."""
    base = bpe_palabras(textos, t_etapa1)
    secs = [base.encode(x).tokens for x in textos]; fusiones = []
    for _ in range(T - t_etapa1):
        cnt = Counter()
        for s in secs: cnt.update(zip(s, s[1:]))
        if not cnt: break
        (a, b), _ = max(cnt.items(), key=lambda kv: (kv[1], kv[0])); fusiones.append((a, b))
        for k, s in enumerate(secs):
            i, out = 0, []
            while i < len(s):
                if i < len(s) - 1 and s[i] == a and s[i + 1] == b: out.append(a + b); i += 2
                else: out.append(s[i]); i += 1
            secs[k] = out
    return base, fusiones

def codifica_super(base, fusiones, texto):
    s = base.encode(texto).tokens
    for a, b in fusiones:
        i, out = 0, []
        while i < len(s):
            if i < len(s) - 1 and s[i] == a and s[i + 1] == b: out.append(a + b); i += 2
            else: out.append(s[i]); i += 1
        s = out
    return s

T_TOTAL, T_ETAPA1 = (2000, 1500) if FAST_DEV_RUN else (16000, 12000)
res_sb = {}
for lang in ["en", "es"]:
    tr = [d["text"] for d in train_docs if d["lang"] == lang]; te = test_by_lang[lang]
    t0 = time.time(); base, fus = superbpe(tr, T_ETAPA1, T_TOTAL); t_tr = time.time() - t0
    bpe = bpe_palabras(tr, T_TOTAL); nbytes = sum(len(x.encode()) for x in te)
    n_bpe = sum(len(bpe.encode(x).tokens) for x in te); n_sup = sum(len(codifica_super(base, fus, x)) for x in te)
    res_sb[lang] = {"bpe": nbytes / n_bpe, "super": nbytes / n_sup, "reduccion": 1 - n_sup / n_bpe, "fusiones": fus, "seg": t_tr}
    print(f"{lang}: bytes/token BPE {nbytes/n_bpe:.2f} -> SuperBPE {nbytes/n_sup:.2f} | {1-n_sup/n_bpe:.1%} menos tokens | entrenar etapa 2: {t_tr:.0f}s | ejemplos: {[a+b for a, b in fus[:6]]}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
x = np.arange(2); w = 0.35
ax[0].bar(x - w / 2, [res_sb[l]["bpe"] for l in ["en", "es"]], w, label=f"BPE (|V|={T_TOTAL})", color=COL["azul"])
ax[0].bar(x + w / 2, [res_sb[l]["super"] for l in ["en", "es"]], w, label=f"SuperBPE (t={T_ETAPA1})", color=COL["naranja"])
ax[0].set_xticks(x); ax[0].set_xticklabels(["inglés", "español"]); ax[0].set_ylabel("Bytes por token (test)"); ax[0].set_title("Misma talla de vocabulario, menos tokens"); ax[0].legend()
for l, c_ in zip(["en", "es"], [COL["verde"], COL["gris"]]):
    n_palabras = [a.count("▁") + b.count("▁") for a, b in res_sb[l]["fusiones"]]
    ax[1].plot(np.arange(len(n_palabras)), np.cumsum([1 if (a.count("▁") + b.count("▁")) >= 2 else 0 for a, b in res_sb[l]["fusiones"]]), label=f"{l}: superpalabras acumuladas", color=c_, lw=2)
ax[1].set_xlabel("Fusión de la etapa 2"); ax[1].set_ylabel("Tokens que cruzan ≥ 1 espacio (acumulado)"); ax[1].set_title("Etapa 2: casi todo son superpalabras"); ax[1].legend()
plt.tight_layout(); plt.show()
print("Superpalabras más frecuentes (ES):", [a + b for a, b in res_sb["es"]["fusiones"][:10]])
print("Superpalabras más frecuentes (EN):", [a + b for a, b in res_sb["en"]["fusiones"][:10]])

> 👀 **Qué debes observar.** **Izquierda:** con el *mismo* tamaño total de vocabulario, SuperBPE cubre más bytes por token (en nuestra escala diminuta la ganancia es de unos pocos %; los autores reportan hasta 33 % con 200 k y gran corpus: la ganancia crece con el vocabulario porque hay más pares frecuentes que fusionar).
> **Derecha:** casi toda la etapa 2 produce superpalabras. Las primeras son **palabras funcionales** frecuentes (`de la`, `en el`, `of the`): no son «semánticas»; en corpus grandes aparecen colocaciones como `a lot of`.
> **Límites:** más memoria/CPU para entrenar el tokenizer, riesgo de superpalabras de *boilerplate* si no deduplicas, y el trabajo reporta caídas puntuales (LAMBADA). Es 🟡.

## 12. Receta: diseñar el vocabulario de Políglota

Un vocabulario se diseña con **tres palancas** y una función de coste:

1. **Mezcla del corpus del tokenizer** (qué proporción de ES, EN, código y, si aplica, otros idiomas). Un corpus «natural» (dominado por inglés) deja pocos merges al resto; suele muestrearse con **temperatura** $p_\ell\propto n_\ell^{\alpha}$ ($\alpha<1$ sube la proporción de idiomas pequeños; XLM-R usa $\alpha=0.3$ para muestrear idiomas al entrenar el modelo).
2. **Tamaño $V$**: se elige en la **rodilla** de la curva *tokens por KB vs $V$* (cuando añadir 1 k tokens ya ahorra < $\varepsilon$ %), acotado por el presupuesto de parámetros (sección 5).
3. **Pre-tokenización** (regex): dígitos, espacios/indentación, mayúsculas (módulo 02/03).

**Función de coste (la que usarás en el proyecto):** con tráfico esperado $w_\ell$ por idioma,
$\;J(V,\text{mezcla})=\sum_\ell w_\ell\cdot\text{tokens por KB}_\ell\;+\;\lambda\cdot V/10^{3}$, y se compara también la **peor prima** entre idiomas (equidad).

Primero medimos el efecto de la **mezcla** con un experimento controlado (mismo $V$, mismo total de texto de entrenamiento, solo cambia la proporción).

In [ ]:
def corpus_mezcla(prop: dict, total_chars: int, semilla: int = SEED) -> list:
    """Muestrea documentos de train por idioma hasta total_chars*prop[idioma] caracteres (con reposición si hace falta)."""
    r = random.Random(semilla); out = []
    for lang, p in prop.items():
        pool = [d["text"] for d in train_docs if d["lang"] == lang]; acc = 0
        while acc < p * total_chars:
            t = r.choice(pool); out.append(t); acc += len(t)
    r.shuffle(out); return out

def bytes_por_token(tok, lang): 
    te = test_by_lang[lang]; return sum(len(x.encode()) for x in te) / sum(len(tok.encode(x).ids) for x in te)

TOTAL = 400_000 if FAST_DEV_RUN else 4_000_000
natural = {"en": 0.80, "es": 0.15, "code": 0.05}
suav = {k: v ** 0.3 / sum(u ** 0.3 for u in natural.values()) for k, v in natural.items()}      # temperatura alpha = 0.3
MEZCLAS = {"natural 80/15/5": natural, f"temperatura α=0.3 ({suav['en']:.0%}/{suav['es']:.0%}/{suav['code']:.0%})": suav,
           "equilibrada 1/3": {"en": 1 / 3, "es": 1 / 3, "code": 1 / 3}, "sesgo ES 20/60/20": {"en": 0.2, "es": 0.6, "code": 0.2}}
res_mix = {}
for nombre, prop in MEZCLAS.items():
    tk = entrena_bpe_bytes(corpus_mezcla(prop, TOTAL), V_PROPIO)
    res_mix[nombre] = {l: bytes_por_token(tk, l) for l in ["es", "en", "code"]}
dmix = pd.DataFrame(res_mix).T; print(dmix.round(2))
fig, ax = plt.subplots(figsize=(9, 4)); dmix[["es", "en", "code"]].plot.bar(ax=ax, rot=15, color=[COL["verde"], COL["azul"], COL["naranja"]])
ax.set_ylabel("Bytes por token en test (mayor = mejor)"); ax.set_title(f"La mezcla del corpus del tokenizer reparte el vocabulario (|V|={V_PROPIO})"); plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** Con el mismo $V$ y el mismo volumen de texto, **quien más pesa en la mezcla se lleva más merges**: la mezcla «natural» (80 % inglés) da el mejor inglés y el peor español; el sesgo al español invierte el orden. La mezcla suavizada (temperatura) y la equilibrada reparten mejor, **a costa de una pequeña pérdida en el idioma dominante**.
> No hay almuerzo gratis: es un reparto de un presupuesto fijo, y la decisión correcta depende del **tráfico** y de la política de equidad del producto (la sección siguiente lo formaliza).

In [ ]:
# Barrido de V con la mezcla suavizada: ¿dónde está la rodilla?  y función de coste J con tráfico esperado
trafico = {"es": 0.5, "en": 0.3, "code": 0.2}          # supuesto de producto (parámetro)
LAMBDA = 3.0                                           # penalización de memoria: "tokens/KB" equivalentes por cada 1 k de vocabulario (parámetro de producto)
VS = [1000, 2000, 4000, 8000, 16000] if FAST_DEV_RUN else [4000, 8000, 16000, 32000, 64000]
corp = corpus_mezcla(suav, TOTAL); filas = []
for V in VS:
    tk = entrena_bpe_bytes(corp, V)
    tpk = {l: 1000 / bytes_por_token(tk, l) for l in ["es", "en", "code"]}               # tokens por KB de texto
    J = sum(trafico[l] * tpk[l] for l in tpk) + LAMBDA * V / 1e3
    filas.append({"V": V, **{f"tpk_{l}": v for l, v in tpk.items()}, "tokens_prom": sum(trafico[l] * tpk[l] for l in tpk), "J": J})
dv = pd.DataFrame(filas).set_index("V"); print(dv.round(2))
mejora = (dv["tokens_prom"].shift(1) - dv["tokens_prom"]) / dv["tokens_prom"].shift(1) * 100
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for l, c_ in zip(["es", "en", "code"], [COL["verde"], COL["azul"], COL["naranja"]]): ax[0].plot(dv.index, dv[f"tpk_{l}"], "o-", label=l, color=c_, lw=2)
ax[0].set_xscale("log"); ax[0].set_xlabel("Tamaño del vocabulario V"); ax[0].set_ylabel("Tokens por KB de texto (menor = mejor)"); ax[0].set_title("Rendimientos decrecientes por idioma"); ax[0].legend()
ax[1].plot(dv.index, dv["J"], "o-", c=COL["rojo"], lw=2); V_opt = int(dv["J"].idxmin()); ax[1].axvline(V_opt, color=COL["gris"], ls="--")
ax[1].set_xscale("log"); ax[1].set_xlabel("Tamaño del vocabulario V"); ax[1].set_ylabel("Coste J = Σ w·tokens/KB + λ·V/1000"); ax[1].set_title(f"Mínimo de J en V = {V_opt}"); plt.tight_layout(); plt.show()
print("mejora % al duplicar V:", mejora.round(1).dropna().to_dict(), "| V* =", V_opt)

> 👀 **Qué debes observar.** **Izquierda:** cada duplicación de $V$ ahorra menos tokens que la anterior (en nuestra corrida: −16 %, −14 %, −10 %, −7 %). Entre 4 k y 16 k el español sigue mejorando más que el inglés (≈ −21 % frente a ≈ −15 %), mientras el código, que necesita más tokens por KB, mejora menos (≈ −11 %).
> **Derecha:** $J$ tiene un mínimo interior: pasado cierto $V$, el coste en parámetros supera el ahorro de tokens (con $\lambda=3$ el mínimo cae en 8 000).
> Ese mínimo **depende de $\lambda$ y del tráfico** (decisiones de producto): cámbialos y verás moverse $V^*$. En un LLM real $\lambda$ sale del presupuesto de parámetros (sección 5) y de las leyes de escala (Tao et al., 2024).

## 13. Resumen de decisiones de diseño

| Decisión | Pregunta que debes poder responder con datos | Herramienta de esta lección |
|---|---|---|
| Tamaño $V$ | ¿dónde está la rodilla *tokens/KB vs V* y cuánto pesan los embeddings? | secciones 5 y 12 |
| Mezcla de idiomas | ¿qué idioma paga más y es aceptable? | prima de paridad (sección 7) y experimento de mezcla |
| Números | ¿dígitos sueltos, grupos de 3 o R2L? | sección 9 |
| Código | ¿se colapsan los runs de espacios? ¿tabs? | sección 9 |
| Especiales y plantilla | ¿una única fuente de verdad para entrenar y servir? | sección 10 |
| ¿Cruzar palabras? | ¿ganancia > complejidad? (🟡) | sección 11 |

## 14. 🏭 En producción

Cifras de vocabulario **a oct-2026** (ver tabla de la sección 6); verifica siempre contra el modelo exacto.

- **OpenAI.** `o200k_base` (GPT-4o) amplió el vocabulario de ≈ 100 k a ≈ 200 k; en nuestra medición reduce la prima de hindi de ≈ 4.9× a ≈ 1.6× y la de ruso de ≈ 2.1× a ≈ 1.25× frente a `cl100k` (mini-conjunto paralelo; repítelo con FLORES+).
- **Meta.** Llama 3 pasó de 32 000 (Llama 2) a 128 256 tokens y reporta una compresión sobre inglés de 3.17 → 3.94 caracteres/token.
- **Mistral.** Tekken (≈ 130 k + ≈ 1 k de control, basado en tiktoken, > 100 idiomas) sustituye al SentencePiece anterior y Mistral reporta ≈ 30 % mejor compresión en código y varios idiomas (según su guía de tokenización).
- **Google.** Gemma usa vocabularios de ≈ 256 k con dígitos separados y bytes; la proporción de parámetros en embeddings es muy alta en los modelos pequeños (gráfico de la sección 5).
- **Cohere.** Command R usa ≈ 255 k tokens con embeddings atados (Land & Bartolo 2024); Cohere publicó además el análisis de tokens sub-entrenados que verás en el módulo 05.
- **Coste real.** Facturación por token + paridad = **prima de precio por idioma** (Ahia et al., 2023). Antes de comprometer un contrato de API para un producto multilingüe, **mide tus textos con el tokenizer del proveedor** y proyecta el coste por idioma.
- **Evaluación independiente.** Compara tokenizers con un corpus que **no** se usó para entrenarlos (FLORES+, por ejemplo) y siempre sobre *contenido paralelo*.

## 15. 🧠 Secretos de la élite

1. **Compara en bits por byte, no por token.** Perplexidad/pérdida por token no es comparable entre vocabularios distintos; Tao et al. (2024) usan una pérdida normalizada por carácter para sus leyes de escala.
2. **El vocabulario óptimo crece con el cómputo, y los modelos reales están por debajo.** Tao et al. predicen ≥ 216 K para Llama2-70B frente a los 32 K reales; los vocabularios 100–260 k de 2024–2026 van en esa dirección (🟡: resultado a ≤ 3 B parámetros).
3. **El «token tax» es un coste de producto medible.** Petrov et al. (2023) encuentran diferencias de hasta 15× entre idiomas en los tokenizers que analizaron, y Ahia et al. (2023) lo convierten en precios. Un equipo con usuarios hispanohablantes debe tener este número en su dashboard.
4. **Los vocabularios de modelo pequeño son «tablas de embeddings».** Pasar de 32 k a 256 k puede mover de ≈ 10 % a ≈ 50 % de los parámetros (gráfico de la sección 5): por eso importa el *weight tying*.
5. **La pre-tokenización de números cambia la aritmética.** Singh & Strouse (2024): tokenización R2L sube la suma de 75.6 % a 97.8 % en GPT-3.5 (8-shot; efecto menor pero presente en GPT-4, marzo 2023); es un cambio de *una línea de regex* con efecto medible.
6. **El espacio en blanco se paga en tokens.** Con GPT-2 la indentación cuesta ≈ 15 tokens extra por línea a 16 espacios; la agrupación de `cl100k`/`o200k` la deja en ≈ 1 (medido arriba, sobre código Python real).
7. **Cambiar el tokenizer de un modelo ya entrenado es caro.** Dagan et al. (2024) reportan que especializar el tokenizer de un modelo preentrenado tras afinarlo con más de 50 B de tokens da ganancias grandes de velocidad y contexto efectivo, pero ese volumen de datos ilustra el coste (módulo 07); por eso se diseña bien a la primera.
8. **Tokenizers que cruzan palabras (SuperBPE) comprimen más con el mismo vocabulario**, pero la evidencia de mejora de modelo viene de un único grupo a 8 B en inglés (🟡).

## 16. ⚠️ Errores comunes

1. Medir la paridad con **texto no paralelo** (los idiomas tienen densidad de información distinta) o con muy pocas frases.
2. **Comparar perplexidades por token** entre tokenizers.
3. Fijar `V` «porque lo usa Llama» sin mirar el presupuesto de parámetros ni el tráfico por idioma.
4. Entrenar el tokenizer con el **corpus natural** (80 % inglés) y sorprenderse del coste en español.
5. Olvidar los **tokens especiales** en el vocabulario del modelo (fila de embedding no inicializada o sin entrenar → módulo 05).
6. Usar una **plantilla de chat distinta** en entrenamiento y en servicio, o duplicar BOS.
7. Cambiar la regex de pre-tokenización de números/espacios **después** de entrenar el modelo.
8. Citar precios y tamaños de vocabulario sin fecha: cambian cada versión.

## 17. Autoevaluación

<details><summary><b>1 · (Cálculo)</b> Un modelo con L = 32, d = 4096 y V = 128 256 sin weight tying: ¿cuántos parámetros hay en embeddings y qué fracción son del total?</summary>

$N_{\text{no-emb}}\approx12\cdot32\cdot4096^2\approx6.44\times10^9$; $N_{\text{emb}}=2\cdot128256\cdot4096\approx1.05\times10^9$; total ≈ 7.49 B → **≈ 14 %** en embeddings. (Es una aproximación genérica: GQA/SwiGLU cambian constantes.)
</details>

<details><summary><b>2 · (Cálculo)</b> Un texto español mide 2 000 bytes. Con β = 4.0 bytes/token cuesta 500 tokens; con β = 3.2 cuesta 625. Si la API cobra 1 USD por millón de tokens y procesas 10 M de textos así al mes, ¿cuánto cuesta cada opción?</summary>

500 tok × 10 M = 5 000 M tok → **5 000 USD**; 625 tok × 10 M = 6 250 M tok → **6 250 USD** (+25 %). Un tokenizer con 25 % más de tokens cuesta 25 % más (y consume 25 % más de ventana).
</details>

<details><summary><b>3 · (Diagnóstico)</b> Tu clasificador multilingüe rinde peor en hindi que en español aunque tiene el mismo nº de ejemplos. El tokenizer es cl100k. ¿Qué miras primero?</summary>

La **prima de tokenización** (hindi ≈ 4.9× en nuestra medición con cl100k): las secuencias hindi son mucho más largas (truncan antes en `max_length`, diluyen la atención) y los tokens por byte son casi bytes sueltos. Mide longitudes de secuencia y tasa de truncamiento por idioma; considera un tokenizer con mejor paridad (o200k, Tekken…) o ampliar `max_length` para hindi.
</details>

<details><summary><b>4 · (Diagnóstico)</b> Tras cambiar la plantilla de chat de tu servicio, la calidad cae 3 puntos sin errores en logs. ¿Causa probable?</summary>

Divergencia entre la plantilla de entrenamiento y la de servicio (un `\n`, un espacio, falta de `add_generation_prompt`, doble BOS). Compara byte a byte la cadena y los ids producidos en servicio con un ejemplo del dataset de post-entrenamiento.
</details>

<details><summary><b>5 · (Transferencia)</b> Políglota añadirá portugués y quiere mantener V ≈ 32 k para un embedder pequeño. ¿Qué haces?</summary>

Mides la prima actual de pt con un corpus paralelo (FLORES+), reentrenas con mezcla suavizada (temperatura) que suba el peso del portugués y barres V (p. ej. 24 k–48 k) con la función de coste J; decides con tráfico esperado y presupuesto de parámetros. Si el vocabulario viejo ya está en producción, evalúas el coste de re-indexar (módulos 07 y 18).
</details>

<details><summary><b>6 · (Transferencia)</b> ¿Adoptarías SuperBPE en Políglota? Argumenta.</summary>

Pros: ≈ 6–7 % menos tokens a igual V en nuestra escala (hasta ≈ 33 % reportado con 200 k), menos coste y más contexto. Contras: evidencia 🟡 (8 B, solo inglés), tokenizer más pesado de entrenar, riesgo de superpalabras de boilerplate, y un buscador con BM25/embeddings por token se beneficia de granularidad de palabra para *matching*. Haría un A/B sobre el golden set antes de adoptarlo.
</details>

<details><summary><b>7 · (Recuerdo)</b> ¿Por qué se compara en «bits por byte» entre tokenizers?</summary>

Porque la pérdida por token depende de cuánto texto cubre cada token; normalizando por bytes/caracteres la comparación es independiente del vocabulario.
</details>

## 18. Referencias

**Papers (verificados)**
- Tao, C. et al. (2024). *Scaling Laws with Vocabulary: Larger Models Deserve Larger Vocabularies.* NeurIPS 2024. <https://arxiv.org/abs/2407.13623>
- Petrov, A., La Malfa, E., Torr, P. y Bibi, A. (2023). *Language Model Tokenizers Introduce Unfairness Between Languages.* NeurIPS 2023. <https://arxiv.org/abs/2305.15425>
- Ahia, O. et al. (2023). *Do All Languages Cost the Same? Tokenization in the Era of Commercial Language Models.* EMNLP 2023. <https://arxiv.org/abs/2305.13707>
- Liu, A., Hayase, J., Hofmann, V., Oh, S., Smith, N. A. y Choi, Y. (2025). *SuperBPE: Space Travel for Language Models.* COLM 2025. <https://arxiv.org/abs/2503.13423>
- Singh, A. K. y Strouse, D. J. (2024). *Tokenization counts: the impact of tokenization on arithmetic in frontier LLMs.* <https://arxiv.org/abs/2402.14903>
- Dagan, G., Synnaeve, G. y Rozière, B. (2024). *Getting the most out of your tokenizer for pre-training and domain adaptation.* <https://arxiv.org/abs/2402.01035>
- Rust, P. et al. (2021). *How Good is Your Tokenizer? On the Monolingual Performance of Multilingual Language Models.* <https://arxiv.org/abs/2012.15613>
- Goldman, O. et al. (2024). *Unpacking Tokenization: Evaluating Text Compression and its Correlation with Model Performance.* <https://arxiv.org/abs/2403.06265>
- Schmidt, C. W. et al. (2024). *Tokenization Is More Than Compression.* <https://arxiv.org/abs/2402.18376>
- Land, S. y Bartolo, M. (2024). *Fishing for Magikarp.* <https://arxiv.org/abs/2405.05417> (tabla de vocabularios)
- Llama Team (2024). *The Llama 3 Herd of Models.* <https://arxiv.org/abs/2407.21783> · Gemma Team (2025). *Gemma 3 Technical Report.* <https://arxiv.org/abs/2503.19786> · Yang et al. (2024). *Qwen2 Technical Report.* <https://arxiv.org/abs/2407.10671>
- Conneau, A. et al. (2019). *Unsupervised Cross-lingual Representation Learning at Scale.* <https://arxiv.org/abs/1911.02116>
- Touvron, H. et al. (2023). *LLaMA.* <https://arxiv.org/abs/2302.13971>

**Datos, librerías y docs**
- FLORES+ (`openlanguagedata/flores_plus`) y `Helsinki-NLP/opus-100` en Hugging Face Datasets; `tiktoken` <https://github.com/openai/tiktoken>; Mistral, guía de tokenización <https://docs.mistral.ai/guides/tokenization/>.

**Siguiente:** módulo 05 (qué falla cuando el vocabulario y los datos del modelo no coinciden: glitch tokens, números, letras, Unicode).